# Qwen2-Audio INT8: kiểm tra chất lượng toàn model (v12 — KAGGLE)

Mục tiêu: đo xem chuỗi INT8 *tĩnh* (cùng quy ước bộ tăng tốc PL) cho đầu ra giống model FP đến mức nào. Mô phỏng GPU (`qint8_sim.py`) INT8×INT8→INT32 chính xác, norm/RoPE/softmax/GELU/SiLU/residual ở FP32; **không phải bit-true với RTL**. Scale tĩnh hiệu chuẩn trên clip thật, chia chéo 2 nhánh; teacher-forced theo chuỗi FP (`repetition_penalty=1.1`). Cấu hình chốt: **P1.5** = scale INT8 theo kênh đầu ra, margin 1,5, fallback hàng c=2, làm mượt offline.

**Khác v11:**
* Chấm điểm có 3 mức: `s1` (đáp án đúng trong **câu đầu**, mức chính), `s2` (2 câu đầu), `any` (bất kỳ đâu, cận trên). v11 chỉ dùng "bất kỳ đâu" nên tính nhầm câu trả lời sai nhưng có nhắc đáp án ở vế khác (vd. Q3 "...surpassing the Nile").
* Chọn câu hỏi **đều theo thứ tự gốc** (`np.linspace`) nên mọi nhóm (địa lý, Việt Nam, khoa học, lịch sử, máy tính/toán, kiến thức chung) đều có mặt kể cả khi `QUIZ_N` < 150.
* Ghi riêng tỉ lệ hàng FP của **prefill** và **decode** khi sinh câu trả lời, để làm rõ vì sao v11 đo ~4,9% khi sinh nhưng ~1,3% khi teacher-forced.
* Tắt TF32 (mô phỏng INT8 chính xác cần fp32 thật).
* Bản **Kaggle (nhẹ)**: 50 clip, 50 câu hỏi chọn đều, chỉ FP32 và INT8 (bỏ mốc bf16 khi sinh câu trả lời; mốc bf16 teacher-forced vẫn đo), không quét cấu hình.
* File wav sinh bằng edge-tts nằm trong `OUT_DIR/gen_wavs` (không ghi vào repo). Kết quả ghi tiếp (resume) trong `OUT_DIR`: chạy lại cell sẽ không sinh lại câu đã xong.

Cách dùng: bật GPU + Internet, **Run All**. Thử nhanh: `N_CLIPS = 2`, `QUIZ_N = 10`.

In [1]:
# ===== 0. Cấu hình =====
PLATFORM = "kaggle"
N_CLIPS = 50          # 14 = chi cau goc; 2 = chay thu nhanh
QUIZ_N = 50          # so cau hoi co dap an (chon deu trong 150; 10 = chay thu)
QUIZ_MODES = ("fp32", "int8")
RUN_SWEEP = False       # cell 12b: quet them cau hinh
QUIZ_MAX_NEW = 300    # tran so token moi cau tra loi; cau nao cham tran se bi danh dau CAP
MAX_NEW = 200
MODEL_ID = "tranquangchung/qwen2-audio-dialogue"
REV = "01061daa248750e56b43801a3c319fb4e6443704"
REPO_URL = "https://github.com/tranquangchung/Audio_Speech_Conversation"
import os
ROOT = "/kaggle/working" if PLATFORM == "kaggle" else "/content"
OUT_DIR = ROOT + "/qint8_report"
if PLATFORM == "colab":
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUT_DIR = "<OUT_DIR>"          # ket qua + cache resume luu tren Drive
    except Exception as e:
        print("Khong gan duoc Google Drive (", type(e).__name__, ") -> luu cuc bo, se MAT khi ngat ket noi:", OUT_DIR)
GEN_DIR = OUT_DIR + "/gen_wavs"
os.makedirs(GEN_DIR, exist_ok=True)
print("OUT_DIR =", OUT_DIR)

OUT_DIR = /kaggle/working/qint8_report


In [2]:
import os, sys, json, time, glob, subprocess, importlib.metadata
need = {"transformers": "4.55.0", "accelerate": "1.10.1", "huggingface-hub": "0.34.4", "librosa": "0.11.0", "soundfile": "0.13.1"}
miss = []
for n, v in need.items():
    try:
        if importlib.metadata.version(n) != v: miss.append(f"{n}=={v}")
    except importlib.metadata.PackageNotFoundError:
        miss.append(f"{n}=={v}")
if miss:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *miss], capture_output=True, text=True)
    print("pip:", miss, "->", r.returncode, r.stderr[-300:])
    print(">>> Neu vua cai transformers moi: Restart kernel roi Run All lai (chi can 1 lan).")
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "Can bat GPU (Settings -> Accelerator)"
gb = torch.cuda.get_device_properties(0).total_memory / 2**30
print(torch.cuda.get_device_name(0), round(gb, 1), "GB")
if gb < 36: print(">>> CANH BAO: model FP32 can ~34 GB + bo nho kich hoat; GPU nay co the khong du (OOM o cell 3). Chon A100/H100 bo nho >= 40 GB.")

pip: ['transformers==4.55.0', 'accelerate==1.10.1', 'huggingface-hub==0.34.4', 'soundfile==0.13.1'] -> 0 res huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.34.4 which is incompatible.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.34.4 which is incompatible.

>>> Neu vua cai transformers moi: Restart kernel roi Run All lai (chi can 1 lan).
torch 2.11.0+cu128 | transformers 4.55.0 | cuda True
Tesla T4 14.6 GB
>>> CANH BAO: model FP32 can ~34 GB + bo nho kich hoat; GPU nay co the khong du (OOM o cell 3). Chon A100/H100 bo nho >= 40 GB.


In [3]:
# ===== 1. Clone repo (14 wav goc) + sinh them wav bang edge-tts cho du N_CLIPS (luu o GEN_DIR) =====
REPO = ROOT + "/Audio_Speech_Conversation"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
BASE_QUESTIONS = [
 "If you had to live in a video game universe for a year, which one would you choose?",
 "Would you rather be able to speak every language fluently or play every musical instrument perfectly?",
 "If animals could talk, which species do you think would be the rudest?",
 "If you found a suitcase with $1 million inside but no ID, would you keep it or turn it in?",
 "If you were a ghost, who would you haunt and how?",
 "Would you rather travel 100 years into the past or 100 years into the future?",
 "If you could only eat one cuisine (Italian, Japanese, Mexican, etc.) for the rest of your life, what would it be?",
 "If you were forced to participate in the Olympics, which sport would you have the best chance of not embarrassing yourself in?",
 "If you could cure one disease or solve one world problem, which would you pick?",
 "If you had to delete all apps from your phone except for three, which ones would you keep?",
 "What is the capital city of Vietnam?",
 "Vietnam is consistently one of the world's top two exporters of which caffeinated commodity?",
 "What is the ancient name of Hanoi, which translates to Ascending Dragon",
 "In Vietnamese culture, what are the specific flowers used to decorate homes during the Tet holiday (Lunar New Year) in the North versus the South?"]
EXTRA_QUESTIONS = [
 "If you could have dinner with any historical figure, who would you invite?",
 "What is one skill you have always wanted to learn but never had the time for?",
 "If you could instantly master any scientific subject, which one would you choose?",
 "Would you rather explore the deepest parts of the ocean or travel into deep space?",
 "If you could teleport anywhere in the world right now for one hour, where would you go?",
 "What is the most memorable book or movie you have ever experienced?",
 "If you had a time machine that only worked once, would you use it?",
 "Which season of the year do you enjoy the most and why?",
 "If you could design your ideal weekend, what activities would it include?",
 "Would you rather live in a bustling modern city or a quiet cabin in the mountains?",
 "If you could switch jobs with anyone for a single day, whose job would you try?",
 "What do you think will be the biggest technological breakthrough in the next ten years?",
 "If you had to teach a class on any topic, what subject would you teach?",
 "Would you rather have a rewind button or a pause button for your life?",
 "If you could bring back one extinct animal species, which one would it be?",
 "What is your favorite way to relax after a long and busy day?",
 "If you could only listen to three musical artists for the rest of your life, who would they be?",
 "Would you rather never be stuck in traffic again or never catch a cold again?",
 "If you could invent a new holiday, what would it celebrate?",
 "What is one piece of advice you would give to someone starting university?",
 "Which planet in our solar system is known as the Red Planet?",
 "What is the largest ocean on Earth?",
 "Who wrote the famous play Romeo and Juliet?",
 "What is the chemical symbol for gold?",
 "Which country is home to the Great Barrier Reef?",
 "How many continents are there on Earth?",
 "What is the fastest land animal in the world?",
 "Which traditional Vietnamese noodle soup is famous around the world?",
 "What is the name of the famous bay in northern Vietnam known for its emerald waters and limestone islands?",
 "In computer science, what does the acronym CPU stand for?",
 "Which gas do plants absorb from the atmosphere during photosynthesis?",
 "What is the tallest mountain in the world above sea level?",
 "Which instrument has eighty-eight keys?",
 "At what temperature in Celsius does water boil at standard atmospheric pressure?",
 "What is the currency used in Japan?",
 "Which organ in the human body is responsible for pumping blood?"]
WAV_DIR = REPO + "/LLM_Question"
ORIG = sorted(p for p in glob.glob(WAV_DIR + "/*.wav") if not os.path.basename(p).startswith("extra_"))
assert len(ORIG) == len(BASE_QUESTIONS), f"repo co {len(ORIG)} wav, can {len(BASE_QUESTIONS)}"
EXTRA_PATHS = [f"{GEN_DIR}/extra_{k:03d}.wav" for k in range(len(BASE_QUESTIONS) + 1, len(BASE_QUESTIONS) + len(EXTRA_QUESTIONS) + 1)]
n_extra = max(0, N_CLIPS - len(ORIG))
todo = [(p, q, len(BASE_QUESTIONS) + 1 + j) for j, (p, q) in enumerate(zip(EXTRA_PATHS[:n_extra], EXTRA_QUESTIONS[:n_extra])) if not os.path.exists(p)]

def tts_all(todo):
    # Ten file co dinh theo so thu tu cau (chay lai khong sinh trung); ghi file tam roi doi ten; thu lai 4 lan moi cau.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "-U", "edge-tts"], check=True)
    import asyncio, threading, edge_tts, librosa, soundfile as sf
    voices = ["en-US-GuyNeural", "en-US-JennyNeural"]
    async def one(path, text, voice):
        tmp_mp3, tmp_wav = path + ".mp3", path + ".part.wav"
        await edge_tts.Communicate(text, voice).save(tmp_mp3)
        y, _ = librosa.load(tmp_mp3, sr=16000, mono=True)
        assert len(y) > 8000, "audio qua ngan"
        sf.write(tmp_wav, y, 16000, format="WAV")
        os.replace(tmp_wav, path); os.remove(tmp_mp3)
    async def main():
        for path, text, k in todo:
            last = None
            for attempt in range(4):
                try:
                    await one(path, text, voices[k % 2]); break
                except Exception as e:
                    last = e; await asyncio.sleep(2 * (attempt + 1))
            else:
                print("LOI tao", os.path.basename(path), type(last).__name__, str(last)[:120])
    err = {}
    def worker():                                        # luong rieng: tranh loi 'event loop is already running' cua Jupyter
        try: asyncio.run(main())
        except Exception as e: err["e"] = e
    th = threading.Thread(target=worker); th.start(); th.join()
    if err: print("LOI edge-tts:", type(err["e"]).__name__, str(err["e"])[:200])

if todo:
    print(f"Dang tao them {len(todo)} file wav (edge-tts, can Internet)...")
    t0 = time.time(); tts_all(todo); print(f"xong [{time.time()-t0:.0f}s]")
PAIRS = [(p, BASE_QUESTIONS[i]) for i, p in enumerate(ORIG)]
PAIRS += [(p, q) for p, q in zip(EXTRA_PATHS, EXTRA_QUESTIONS) if os.path.exists(p)]
PAIRS = PAIRS[:N_CLIPS]
WAVS, QUESTIONS = [p for p, _ in PAIRS], [q for _, q in PAIRS]
N_ORIG = min(len(ORIG), len(PAIRS))
print(len(WAVS), "file wav (", N_ORIG, "goc +", len(WAVS) - N_ORIG, "moi) / yeu cau", N_CLIPS)
if len(WAVS) < N_CLIPS: print("CANH BAO: thieu", N_CLIPS - len(WAVS), "file (edge-tts loi?). Tiep tuc voi so clip hien co.")
assert len(WAVS) >= 2

Cloning into '/kaggle/working/Audio_Speech_Conversation'...


Dang tao them 36 file wav (edge-tts, can Internet)...
xong [35s]
50 file wav ( 14 goc + 36 moi) / yeu cau 50


## 2. Module mô phỏng `qint8_sim.py`

In [4]:
%%writefile qint8_sim.py
"""
qint8_sim.py - Mo phong INT8 (PL + CPU) cho Qwen2-Audio tren GPU bang torch.

Muc dich: do CHAT LUONG (khop token so voi model FP) cua chuoi INT8 "tinh" truoc khi dua len board.
Day la mo phong, KHONG phai bit-true voi RTL: phep nhan tich luy so nguyen la chinh xac (INT8xINT8 -> INT32),
requant dung cung cong thuc (mult, shift) voi qffn_common.make_qparams / requant_golden, nhung buoc
nhan he so diem (QK) va softmax/AV dung float32 nen co the lech 1 LSB o vai gia tri hoa tie.

Cac tensor quan tam (site):
  <ten>.in   dau vao GEMM (luong tu hoa tinh, doi xung, per-tensor)
  <ten>.out  dau ra GEMM sau requant INT8 (per-tensor), tru lm_head khi lm_out == 'int32'
  <attn>.q, <attn>.k  (chi LLM, sau RoPE)   <attn>.score  (diem QK sau requant INT8, khi score_mode == 'int8')
Che do FALLBACK HANG NGOAI LAI (QS.row_fb): vai token (token dau chuoi `<|audio_bos|>`, vai `<|AUDIO|>`) co kich hoat
lon bat thuong (massive activations). Khi hieu chuan, cac hang do (max hang > row_c * trung vi max hang) bi loai khoi thong ke
scale; khi chay, hang nao se bao hoa (|x| > 127.5*scale vao, hoac dau ra cham 127/-128) duoc tinh bang FP tren CPU voi trong so
INT8 giai luong tu (W8*sw), khong qua PL. Cac vi tri bao hoa cung duoc tinh FP trong attention (cot K/V, hang Q).
Trong so: per-channel (moi kenh dau ra mot scale). Phan khong nhan ma tran (norm, RoPE, softmax, GELU/SiLU, cong
residual, embedding) chay float32 nhu tren CPU.
"""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
try:
    torch.set_float32_matmul_precision("highest")
except Exception:
    pass

PCTS = (100.0, 99.999, 99.99, 99.9, 99.5)
KCHUNK = 1024  # 1024*127*128 < 2^24  => tong tung khoi float32 la so nguyen chinh xac


class QState:
    def __init__(self):
        self.mode = "off"          # 'off' (FP goc) | 'calib' (FP + ghi thong ke) | 'quant' (INT8)
        self.scales = {}           # site -> scale kich hoat
        self.stats = {}            # site -> [tensor(len(PCTS))]  (thong ke cua clip hien tai)
        self.valid = {}            # 'mel','enc','proj' -> so hang hop le
        self.fp_layers = set()     # khoa lop giu FP, vd 'enc.5', 'llm.0', 'conv1', 'proj', 'lm_head'
        self.score_mode = "int8"   # 'int8' | 'int32'
        self.lm_out = "int32"      # 'int8' | 'int32'
        self.collect = None        # dict: ten lop -> tensor dau ra (cpu) de do loi tung lop
        self.row_fb = False        # fallback FP cho hang ngoai lai
        self.row_c = 10.0          # hieu chuan: hang ngoai lai neu max hang > row_c * trung vi(max hang)
        self.fb_w = "int8"         # trong so cho hang FP: 'int8' (W8*sw) | 'fp' (trong so goc, can tren)
        self.pos_flag = {}         # layer_key -> bool [B,T]: vi tri ngoai lai cua q/k/v_proj lop do
        self.fb = {}               # ten -> [so hang/vi tri FP, tong] (dem trong calib va quant)
        self.pf_cache = {}         # layer_key -> bool [B,S]: co vi tri ngoai lai cua toan bo khoa K/V (cho decode co KV cache)
        self.int32_ops = ()        # hau to ten GEMM tra ve INT32 (PS giai luong tu bang FP, khong requant INT8), '*' = tat ca
        self.out_pc = False        # dau ra INT8 dung scale THEO KENH dau ra (requant PL da co mult/shift theo kenh; PS giai luong tu bang vector)
        self.fb_phase = {}         # {'pre': [fp, tong], 'dec': [fp, tong]}: hang FP tach theo prefill (nhieu hang) va decode (1 hang/buoc), chi o che do quant, khong tinh attention
        self.sat = "edge"          # co bao hoa dau ra INT8: 'edge' = ket qua ==127/-128 (cu; voi scale theo kenh = max hieu chuan thi bao nhieu hang vo ly) | 'clip' = gia tri that vuot 127.5 (bi cat that)
        self.attn_fp = ()          # chan doan attention: tap con cua {'q','k','v','p'} giu FP (khong luong tu toan hang do)


QS = QState()


# ----------------------------------------------------------------------------- thong ke
@torch.no_grad()
def _stat(x):
    a = x.detach().abs().float().flatten()
    n = a.numel()
    if n == 0:
        return torch.zeros(len(PCTS))
    mx = a.max().view(1)
    step = max(1, n // 2_000_000)
    if step > 1 and step % 2 == 0:
        step += 1
    if step > 1:
        a = a[::step]
    qs = torch.tensor([p / 100.0 for p in PCTS[1:]], device=a.device, dtype=torch.float32)
    q = torch.quantile(a, qs)
    return torch.cat([mx, q]).cpu()


def record(site, x):
    QS.stats.setdefault(site, []).append(_stat(x))


def record_vec(site, y2):
    """Thong ke theo kenh dau ra: max |y| tung kenh cua cac hang y2 [R,N]."""
    QS.stats.setdefault(site, []).append(y2.detach().abs().amax(0).float().cpu())


def start_calib():
    QS.stats = {}
    QS.fb = {}
    QS.pos_flag = {}
    QS.pf_cache = {}


def reset_fb():
    QS.fb = {}
    QS.fb_phase = {}
    QS.pos_flag = {}
    QS.pf_cache = {}


def _note(name, nfb, n):
    a = QS.fb.setdefault(name, [0, 0])
    a[0] += int(nfb)
    a[1] += int(n)
    if QS.mode == "quant" and not name.endswith(".attn"):
        b = QS.fb_phase.setdefault("dec" if int(n) == 1 else "pre", [0, 0])
        b[0] += int(nfb)
        b[1] += int(n)


def fb_split():
    """Ti le hang FP (%) cua prefill va decode: {'pre': (%, tong hang), 'dec': (%, tong hang)}."""
    return {k: (100.0 * v[0] / max(v[1], 1), v[1]) for k, v in QS.fb_phase.items()}


def fb_report(top=12, prefix=None):
    """In ti le hang/vi tri duoc chay FP (QS.fb tich luy tu luc reset)."""
    rows = [(k, v[0], v[1]) for k, v in QS.fb.items() if v[1] > 0 and (prefix is None or k.startswith(prefix))]
    tot_f, tot_n = sum(r[1] for r in rows), sum(r[2] for r in rows)
    print(f"  FP-hang tong: {tot_f}/{tot_n} = {100.0 * tot_f / max(tot_n, 1):.2f}%")
    for k, f, n in sorted(rows, key=lambda r: -r[1] / r[2])[:top]:
        print(f"    {k:22s} {f:6d}/{n:<7d} = {100.0 * f / n:6.2f}%")
    return rows


def _row_keep(x2):
    """Hieu chuan: hang binh thuong = max hang <= row_c * trung vi(max hang)."""
    r = x2.detach().abs().amax(1).float()
    med = r.median()
    if float(med) <= 0:
        return torch.ones_like(r, dtype=torch.bool)
    return r <= QS.row_c * med


def stats_of_clip():
    """Gop thong ke cua 1 clip: site -> tensor[len(PCTS)] (lay max neu site goi nhieu lan)."""
    return {k: torch.stack(v).max(0).values for k, v in QS.stats.items()}


def group_of(site):
    return "llm" if (site.startswith("llm.") or site.startswith("lm_head")) else "enc"


def build_scales(clip_stats, pct_enc=100.0, pct_llm=100.0, pct_map=None, agg="max", pc_margin=1.0):
    """clip_stats: list cac dict (site -> tensor). pct_map: list (chuoi con, pct) uu tien truoc nhom mac dinh."""
    sites = set()
    for d in clip_stats:
        sites |= set(d.keys())
    out = {}
    for s in sorted(sites):
        if s.endswith(".outc"):                          # scale theo kenh dau ra: max qua cac clip / 127
            mx = torch.stack([d[s] for d in clip_stats if s in d]).max(0).values
            out[s] = torch.where(mx > 0, mx * pc_margin / 127.0, torch.ones_like(mx))     # pc_margin > 1: du dia cho kenh co max thuc te nhinh hon hieu chuan
            continue
        pct = pct_enc if group_of(s) == "enc" else pct_llm
        for sub, p in (pct_map or []):
            if sub in s:
                pct = p
                break
        pi = PCTS.index(pct)
        vals = torch.stack([d[s][pi] for d in clip_stats if s in d])
        amax = float(vals.max() if agg == "max" else vals.mean())
        out[s] = amax / 127.0 if amax > 0 else 1.0
    return out


# ----------------------------------------------------------------------------- so hoc so nguyen
def int_mm(x8, W8):
    """x8 [M,K] float32 (so nguyen), W8 [N,K] int8 -> [M,N] float64 (so nguyen chinh xac)."""
    M, K = x8.shape
    acc = None
    for k0 in range(0, K, KCHUNK):
        part = x8[:, k0:k0 + KCHUNK] @ W8[:, k0:k0 + KCHUNK].float().t()
        part = part.double()
        acc = part if acc is None else acc + part
    return acc


def qparams(real):
    """Vector hoa qffn_common.make_qparams: real [N] float64 -> (mult int64, shift int64)."""
    real = real.double()
    shift = torch.clamp(torch.floor(torch.log2((2 ** 31 - 1) / real)), max=62.0)
    mult = torch.round(real * torch.pow(2.0, shift))
    for _ in range(3):
        bad = (mult > 2 ** 31 - 1) & (shift > 0)
        shift = torch.where(bad, shift - 1, shift)
        mult = torch.round(real * torch.pow(2.0, shift))
    assert bool((shift >= 0).all()) and bool((mult >= 1).all()), "he so requant ngoai pham vi"
    return mult.long(), shift.long()


def requant(acc, mult, shift, bias_i=None, zp=0):
    """acc float64 [M,N] (so nguyen). Tra ve int64 da clamp [-128,127]: ((acc+bias)*mult + rnd) >> shift + zp."""
    t = acc.long()
    if bias_i is not None:
        t = t + bias_i
    rnd = torch.where(shift > 0, torch.bitwise_left_shift(torch.ones_like(shift), (shift - 1).clamp(min=0)),
                      torch.zeros_like(shift))
    v = torch.bitwise_right_shift(t * mult + rnd, shift) + zp
    return torch.clamp(v, -128, 127)


# ----------------------------------------------------------------------------- GEMM co luong tu
class QGemm(nn.Module):
    """Co so cho QLinear / QConv1d: trong so INT8 per-channel + quy trinh luong tu."""

    ROWFB = True        # QConv1d tat

    def __init__(self, name, layer_key, weight_nk, bias):
        super().__init__()
        self.name = name
        self.layer_key = layer_key
        self.valid_key = None
        self.last_fb = None
        W = weight_nk.detach()
        N, K = W.shape
        W8 = torch.empty((N, K), dtype=torch.int8, device=W.device)
        sw = torch.empty(N, dtype=torch.float64, device=W.device)
        for n0 in range(0, N, 8192):
            w = W[n0:n0 + 8192].double()
            amax = w.abs().amax(dim=1)
            s = torch.where(amax > 0, amax / 127.0, torch.ones_like(amax))
            W8[n0:n0 + 8192] = torch.clamp(torch.round(w / s[:, None]), -128, 127).to(torch.int8)
            sw[n0:n0 + 8192] = s
        self.register_buffer("W8", W8, persistent=False)
        self.register_buffer("sw", sw, persistent=False)
        self.bias = None if bias is None else bias.detach().double()
        self._cache = None

    @property
    def fp(self):
        return self.layer_key in QS.fp_layers or self.name in QS.fp_layers

    @property
    def s_out(self):
        return QS.scales[self.name + ".out"]

    def sout_vec(self):
        """Scale dau ra theo kenh (tensor float64 tren thiet bi) neu QS.out_pc, nguoc lai None."""
        if not QS.out_pc:
            return None
        src = QS.scales[self.name + ".outc"]
        if getattr(self, "_ov", None) is None or self._ov[0] is not src:
            self._ov = (src, src.to(self.W8.device).double())
        return self._ov[1]

    def vscale(self):
        """Scale dau ra dung cho V trong attention: vector [N] float32 (out_pc) hoac so vo huong."""
        v = self.sout_vec()
        return self.s_out if v is None else v.float()

    def _qp(self, sin, sout):
        key = (sin, id(sout) if torch.is_tensor(sout) else sout)
        if self._cache is None or self._cache[0] != key:
            mult, shift = qparams(sin * self.sw / sout)
            bi = None
            if self.bias is not None:
                bi = torch.clamp(torch.round(self.bias / (sin * self.sw)), -(2 ** 31), 2 ** 31 - 1).long()
            self._cache = (key, mult, shift, bi)
        return self._cache[1:]

    def _wfp(self):
        raise NotImplementedError

    def _fp_rows(self, xf):
        """Hang ngoai lai tren CPU: x FP32 @ W (INT8 giai luong tu, hoac FP goc neu QS.fb_w == 'fp') + bias."""
        N = self.W8.shape[0]
        if QS.fb_w == "fp":
            y = xf @ self._wfp().float().t()
        else:
            y = torch.empty((xf.shape[0], N), dtype=torch.float32, device=xf.device)
            for n0 in range(0, N, 8192):
                Wd = self.W8[n0:n0 + 8192].float() * self.sw[n0:n0 + 8192].float()[:, None]
                y[:, n0:n0 + 8192] = xf @ Wd.t()
        if self.bias is not None:
            y = y + self.bias.float()
        return y

    def gemm(self, x2d, int32_out=False):
        """x2d [M,K] float -> [M,N] float32 (da dequant)."""
        sin = QS.scales[self.name + ".in"]
        xf = x2d.float()
        use_fb = QS.row_fb and self.ROWFB
        x8 = torch.clamp(torch.round(xf / sin), -128, 127)
        acc = int_mm(x8, self.W8)
        fb = (xf.abs().amax(1) > 127.5 * sin) if use_fb else None
        if int32_out:
            if self.bias is not None:
                acc = acc + torch.clamp(torch.round(self.bias / (sin * self.sw)), -(2 ** 31), 2 ** 31 - 1)
            y = (acc * (sin * self.sw)).float()
        else:
            sout = self.sout_vec()
            if sout is None:
                sout = QS.scales[self.name + ".out"]
            mult, shift, bi = self._qp(sin, sout)
            v = requant(acc, mult, shift, bi)
            if use_fb:
                if QS.sat == "clip":
                    raw = (acc + (bi if bi is not None else 0)) * (sin * self.sw / sout)
                    fb = fb | (raw.abs() > 127.5).any(dim=1)
                else:
                    fb = fb | ((v == 127) | (v == -128)).any(dim=1)
            y = v.float() * (sout.float() if torch.is_tensor(sout) else sout)
        if use_fb:
            self.last_fb = fb
            n = QS.valid.get(self.valid_key) if self.valid_key else None
            nv = fb.numel() if n is None else min(n, fb.numel())
            _note(self.name, fb[:nv].sum(), nv)
            if bool(fb.any()):
                y[fb] = self._fp_rows(xf[fb])
        return y


def _rows(x, key):
    n = QS.valid.get(key) if key else None
    return x if n is None else x[:, :n]


class QLinear(QGemm):
    def __init__(self, name, layer_key, lin, valid_key=None):
        super().__init__(name, layer_key, lin.weight, lin.bias)
        self.lin = lin
        self.valid_key = valid_key
        self._isq = name.endswith(".q_proj")
        self._isqkv = name.endswith((".q_proj", ".k_proj", ".v_proj"))

    @property
    def weight(self):                 # HF doc .weight o mot so cho
        return self.lin.weight

    def _wfp(self):
        return self.lin.weight

    def _set_pf(self, fbfull):
        """fbfull bool [B,T] hoac None. q_proj dat lai, k/v_proj hop vao (de attention biet vi tri ngoai lai)."""
        if not self._isqkv:
            return
        cur = None if self._isq else QS.pos_flag.get(self.layer_key)
        if fbfull is not None:
            cur = fbfull if cur is None else (cur | fbfull)
        QS.pos_flag[self.layer_key] = cur

    def forward(self, x):
        if self._isq:
            QS.pos_flag[self.layer_key] = None
        if QS.mode == "off" or self.fp:
            return self.lin(x)
        if QS.mode == "calib":
            if QS.row_fb and self.ROWFB:
                xs = _rows(x, self.valid_key)                    # [B,n,K]
                B_, n_, K_ = xs.shape
                x2 = xs.reshape(-1, K_)
                keep = _row_keep(x2)
                y = self.lin(x)
                ys = _rows(y, self.valid_key).reshape(-1, y.shape[-1])
                record(self.name + ".in", x2[keep])
                record(self.name + ".out", ys[keep])
                if QS.out_pc:
                    record_vec(self.name + ".outc", ys[keep])
                _note(self.name, (~keep).sum(), keep.numel())
                if self._isqkv:
                    pf = torch.zeros(x.shape[0], x.shape[1], dtype=torch.bool, device=x.device)
                    pf[:, :n_] = (~keep).view(B_, n_)
                    self._set_pf(pf)
                return y
            record(self.name + ".in", _rows(x, self.valid_key))
            y = self.lin(x)
            record(self.name + ".out", _rows(y, self.valid_key))
            if QS.out_pc:
                ym = _rows(y, self.valid_key)
                record_vec(self.name + ".outc", ym.reshape(-1, ym.shape[-1]))
            return y
        shp = x.shape
        int32 = (self.name == "lm_head" and QS.lm_out == "int32") or any(
            o == "*" or self.name.endswith(o) for o in QS.int32_ops)
        y = self.gemm(x.reshape(-1, shp[-1]), int32_out=int32)
        if QS.row_fb and self._isqkv and self.last_fb is not None:
            self._set_pf(self.last_fb.view(shp[0], shp[1]))
        return y.reshape(*shp[:-1], -1)


class QConv1d(QGemm):
    ROWFB = False

    def __init__(self, name, layer_key, conv, key_in, key_out):
        w = conv.weight.detach()                      # [out, C, ks]
        super().__init__(name, layer_key, w.permute(0, 2, 1).reshape(w.shape[0], -1), conv.bias)
        self.conv = conv
        self.ks = conv.kernel_size[0]
        self.stride = conv.stride           # tuple: encoder doc self.conv1.stride[0]
        self.pad = conv.padding[0]
        self.key_in, self.key_out = key_in, key_out

    @property
    def weight(self):
        return self.conv.weight

    def _wfp(self):
        w = self.conv.weight
        return w.permute(0, 2, 1).reshape(w.shape[0], -1)

    def _im2col(self, x):                              # x [B,C,T] -> [B,Tout,ks*C], chi so k*C + c
        xp = F.pad(x, (self.pad, self.pad))
        st = self.stride[0]
        tout = (xp.shape[-1] - self.ks) // st + 1
        cols = [xp[:, :, k:k + st * (tout - 1) + 1:st].transpose(1, 2) for k in range(self.ks)]
        return torch.cat(cols, dim=-1)

    def forward(self, x):
        if QS.mode == "off" or self.fp:
            return self.conv(x)
        if QS.mode == "calib":
            n = QS.valid.get(self.key_in)
            record(self.name + ".in", x[:, :, :n] if n else x)
            y = self.conv(x)
            m = QS.valid.get(self.key_out)
            record(self.name + ".out", y[:, :, :m] if m else y)
            if QS.out_pc:
                yc = y[:, :, :m] if m else y
                record_vec(self.name + ".outc", yc.transpose(1, 2).reshape(-1, yc.shape[1]))
            return y
        A = self._im2col(x)
        B_, T_, K_ = A.shape
        y = self.gemm(A.reshape(-1, K_)).reshape(B_, T_, -1)
        return y.transpose(1, 2).contiguous()


# ----------------------------------------------------------------------------- attention
def _chunked_av(P8, V8):
    """P8 [B,H,T,S] , V8 [B,H,S,d] (so nguyen, float32) -> float64 [B,H,T,d] chinh xac."""
    S_ = P8.shape[-1]
    acc = None
    for s0 in range(0, S_, KCHUNK):
        part = (P8[..., s0:s0 + KCHUNK] @ V8[..., s0:s0 + KCHUNK, :]).double()
        acc = part if acc is None else acc + part
    return acc


def _score_sel(S_, mask, kind, pf=None):
    """Chon phan tu hop le de lay thong ke diem (bo cac hang/cot o vi tri ngoai lai pf [B,T])."""
    if kind == "enc":
        n = QS.valid.get("enc")
        S_ = S_[:, :, :n, :n]
        if pf is not None and bool(pf[:, :n].any()):
            keep = ~pf[0, :n]
            S_ = S_[:, :, keep][:, :, :, keep]
        return S_
    if mask is None:
        ok = torch.ones_like(S_, dtype=torch.bool)
    else:
        ok = (mask[:, :, :, :S_.shape[-1]] > -1e30).expand_as(S_)
    if pf is not None and bool(pf.any()):
        ok = ok & ~(pf[:, None, :, None] | pf[:, None, None, :])
    return S_[ok]


def make_attention(orig, kind):
    def fn(module, query, key, value, attention_mask, *args, **kwargs):
        lk = module.layer_key
        if QS.mode == "off" or lk in QS.fp_layers or module.qname in QS.fp_layers:
            return orig(module, query, key, value, attention_mask, *args, **kwargs)
        scaling = kwargs.get("scaling", None)
        if kind == "enc":
            scaling = 1.0                                  # q da nhan module.scaling truoc khi goi
        elif scaling is None:
            scaling = query.size(-1) ** -0.5
        rep = query.shape[1] // key.shape[1]
        mask = attention_mask
        S_len = key.shape[-2]
        mslice = None if mask is None else mask[:, :, :, :S_len]

        fbm = QS.row_fb
        pf = QS.pos_flag.get(lk) if fbm else None            # [B,T]: vi tri cua cac hang query hien tai
        pfk = None                                           # [B,S]: vi tri cua toan bo khoa/gia tri (ke ca KV cache)
        if pf is not None and pf.shape[1] != query.shape[2]:
            pf = None
        if pf is not None:
            if pf.shape[1] == key.shape[2]:                  # prefill / teacher-forced
                pfk = pf
            else:                                            # decode: noi them vao co cua cac vi tri truoc
                prev = QS.pf_cache.get(lk)
                pfk = torch.cat([prev, pf], 1) if (prev is not None and prev.shape[1] + pf.shape[1] == key.shape[2]) else None
            QS.pf_cache[lk] = pfk
        if QS.mode == "calib":
            k = key.repeat_interleave(rep, 1) if rep > 1 else key
            v = value.repeat_interleave(rep, 1) if rep > 1 else value
            if kind == "llm":
                if pf is not None and bool(pf.any()):
                    kp = ~pf[0]
                    record(module.qname + ".q", query[:, :, kp])
                    record(module.qname + ".k", key[:, :, kp])
                else:
                    record(module.qname + ".q", query)
                    record(module.qname + ".k", key)
            sc = torch.matmul(query, k.transpose(2, 3)) * scaling
            record(module.qname + ".score", _score_sel(sc, mask, kind, pf))
            if pf is not None:
                nv = QS.valid.get("enc") if kind == "enc" else pf.shape[1]
                _note(module.qname, pf[0, :nv].sum(), nv)
            if mslice is not None:
                sc = sc + mslice
            p = F.softmax(sc, dim=-1, dtype=torch.float32).to(query.dtype)
            return torch.matmul(p, v).transpose(1, 2).contiguous(), None

        # ---- quant
        v_scale = module.v_proj.vscale()
        if kind == "enc":
            sq = module.q_proj.s_out * module.scaling
            sk = module.k_proj.s_out
            fac = sq * sk                                  # scaling da nam trong sq
        else:
            sq = QS.scales[module.qname + ".q"]
            sk = QS.scales[module.qname + ".k"]
            fac = sq * sk * scaling
        k = key.repeat_interleave(rep, 1) if rep > 1 else key
        v = value.repeat_interleave(rep, 1) if rep > 1 else value
        if torch.is_tensor(v_scale):                       # scale V theo kenh: [1,H,1,d]
            v_scale = v_scale.view(1, value.shape[1], 1, value.shape[3])
            v_scale = v_scale.repeat_interleave(rep, 1) if rep > 1 else v_scale
        af = QS.attn_fp
        q8 = query.float() / sq if "q" in af else torch.clamp(torch.round(query.float() / sq), -128, 127)
        k8 = k.float() / sk if "k" in af else torch.clamp(torch.round(k.float() / sk), -128, 127)
        v8 = v.float() / v_scale if "v" in af else torch.clamp(torch.round(v.float() / v_scale), -128, 127)
        acc = torch.matmul(q8, k8.transpose(2, 3))         # tong <= 64*128*128 < 2^24: chinh xac
        if QS.score_mode == "int8":
            ssc = QS.scales[module.qname + ".score"]
            sc = torch.clamp(torch.round(acc * (fac / ssc)), -128, 127) * ssc
        else:
            sc = acc * fac
        kcol = None
        if fbm:
            # vi tri bao hoa: Q (hang), K/V (cot) -> tinh FP tren CPU
            qrow = query.float().abs().amax(-1) > 127.5 * sq                       # [B,H,T]
            kcol = (k.float().abs().amax(-1) > 127.5 * sk) | ((v.float().abs() > 127.5 * v_scale).any(-1))   # [B,H,S]
            if pf is not None:
                qrow = qrow | pf[:, None, :]
            if pfk is not None:
                kcol = kcol | pfk[:, None, :]
            nv = QS.valid.get("enc") if kind == "enc" else qrow.shape[-1]
            _note(module.qname, (qrow.any(1) | kcol.any(1))[0, :nv].sum(), nv)
            if bool(qrow.any()) or bool(kcol.any()):
                fpm = qrow[..., :, None] | kcol[..., None, :]
                sc_fp = torch.matmul(query.float(), k.float().transpose(2, 3)) * scaling
                sc = torch.where(fpm, sc_fp, sc)
            else:
                kcol = None
        if mslice is not None:
            sc = sc + mslice
        p = F.softmax(sc, dim=-1, dtype=torch.float32)
        pn = p if kcol is None else p.masked_fill(kcol[..., None, :], 0.0)   # cot FP khong tham gia scale cua P
        sp = (pn.amax(dim=-1, keepdim=True) / 127.0).clamp_min(1e-30)
        p8 = pn / sp if "p" in af else torch.round(pn / sp)
        o = _chunked_av(p8, v8) * sp.double() * (v_scale.double() if torch.is_tensor(v_scale) else v_scale)
        if kcol is not None:
            pfp = p.masked_fill(~kcol[..., None, :], 0.0)
            o = o + torch.matmul(pfp, v.float()).double()
        return o.float().transpose(1, 2).contiguous(), None
    return fn


# ----------------------------------------------------------------------------- cai dat vao model
_ORIG = {}


def install(model):
    """Boc cac GEMM cua model bang QLinear/QConv1d va thay ham attention. Goi 1 lan; model o che do FP giu nguyen ket qua."""
    import transformers.models.qwen2_audio.modeling_qwen2_audio as qa
    import transformers.models.qwen2.modeling_qwen2 as q2
    at = model.audio_tower
    lm = model.language_model
    assert at.config._attn_implementation == "eager" and lm.config._attn_implementation == "eager", \
        "can attn_implementation='eager' cho ca audio_tower va language_model"
    if "qa" not in _ORIG:
        _ORIG["qa"], _ORIG["q2"] = qa.eager_attention_forward, q2.eager_attention_forward
    qa.eager_attention_forward = make_attention(_ORIG["qa"], "enc")
    q2.eager_attention_forward = make_attention(_ORIG["q2"], "llm")

    at.conv1 = QConv1d("conv1", "conv1", at.conv1, "mel", "mel")
    at.conv2 = QConv1d("conv2", "conv2", at.conv2, "mel", "enc")
    for i, L in enumerate(at.layers):
        lk = f"enc.{i}"
        a = L.self_attn
        for nm in ("q_proj", "k_proj", "v_proj", "out_proj"):
            setattr(a, nm, QLinear(f"{lk}.{nm}", lk, getattr(a, nm), "enc"))
        L.fc1 = QLinear(f"{lk}.fc1", lk, L.fc1, "enc")
        L.fc2 = QLinear(f"{lk}.fc2", lk, L.fc2, "enc")
        a.qname, a.layer_key = f"{lk}.attn", lk
        L.qname = lk
        L.register_forward_hook(_collect_hook("enc"))
    lin = model.multi_modal_projector.linear
    model.multi_modal_projector.linear = QLinear("proj", "proj", lin, "proj")
    for j, L in enumerate(lm.model.layers):
        lk = f"llm.{j}"
        a, m = L.self_attn, L.mlp
        for nm in ("q_proj", "k_proj", "v_proj", "o_proj"):
            setattr(a, nm, QLinear(f"{lk}.{nm}", lk, getattr(a, nm)))
        for nm in ("gate_proj", "up_proj", "down_proj"):
            setattr(m, nm, QLinear(f"{lk}.{nm}", lk, getattr(m, nm)))
        a.qname, a.layer_key = f"{lk}.attn", lk
        L.qname = lk
        L.register_forward_hook(_collect_hook("llm"))
    lm.lm_head = QLinear("lm_head", "lm_head", lm.lm_head)
    return model


def _collect_hook(kind):
    def hook(mod, inp, out):
        if QS.collect is None:
            return
        h = out[0] if isinstance(out, tuple) else out
        if kind == "enc":
            n = QS.valid.get("enc")
            h = h[:, :n]
        QS.collect[mod.qname] = h.detach().float().cpu()
    return hook


def set_valid(feature_attention_mask):
    L = int(feature_attention_mask.sum(-1)[0])
    L2 = (L - 1) // 2 + 1
    Lp = (L2 - 2) // 2 + 1
    QS.valid = {"mel": L, "enc": L2, "proj": Lp}
    return QS.valid


# ----------------------------------------------------------------------------- tien ich do chat luong
def apply_penalty(logits_row, history_ids, penalty):
    idx = torch.unique(history_ids)
    v = logits_row[idx]
    logits_row[idx] = torch.where(v > 0, v / penalty, v * penalty)
    return logits_row


@torch.no_grad()
def forward_logits(model, inputs, gen_ids):
    """Teacher-forced: tra ve (logits [n_gen,V] cho cac vi tri du doan gen_ids, input_len)."""
    ids = inputs["input_ids"]
    n0 = ids.shape[1]
    full = torch.cat([ids, gen_ids.view(1, -1).to(ids.device)], dim=1)
    out = model(input_ids=full, attention_mask=torch.ones_like(full),
                input_features=inputs["input_features"], feature_attention_mask=inputs["feature_attention_mask"],
                use_cache=False)
    lg = out.logits[0, n0 - 1: full.shape[1] - 1].float()
    return lg, n0, full[0]


def compare_logits(lf, lq, full_ids, n0, penalty=1.1):
    """lf, lq [n,V] (logit tho). Ap dung cung phat lap lai cho 2 ben (lich su = prompt + token da sinh)."""
    n = lf.shape[0]
    lf = lf.to(lq.device)
    agree = []
    rank5 = []
    for t in range(n):
        hist = full_ids[:n0 + t]
        a = apply_penalty(lf[t].clone(), hist, penalty)
        b = apply_penalty(lq[t].clone(), hist, penalty)
        agree.append(bool(a.argmax() == b.argmax()))
        rank5.append(bool((b.topk(5).indices == a.argmax()).any()))
    agree = torch.tensor(agree)
    cos = F.cosine_similarity(lf, lq, dim=-1).mean().item()
    pf, lpq = F.softmax(lf, -1), F.log_softmax(lq, -1)
    kl = (pf * (torch.log(pf.clamp_min(1e-30)) - lpq)).sum(-1).mean().item()
    first = int((~agree).nonzero()[0]) if (~agree).any() else n
    return {"n": n, "top1": agree.float().mean().item(), "top5": torch.tensor(rank5).float().mean().item(),
            "cos": cos, "kl": kl, "first_div": first}

Writing qint8_sim.py


In [5]:
# ===== 3. Nạp model FP32 (eager attention) + processor =====
import gc
for _n in ("model", "REF", "STATS", "STATS_C", "STATS_NOSMOOTH", "CH", "lf0", "lf1"):
    globals().pop(_n, None)                                  # giai phong GPU neu chay lai o nay trong kernel cu
gc.collect(); torch.cuda.empty_cache()
print("GPU dang giu truoc khi nap: %.2f GB (phai gan 0)" % (torch.cuda.memory_allocated() / 2**30))
import numpy as np, librosa
from transformers import AutoProcessor, Qwen2AudioForConditionalGeneration
processor = AutoProcessor.from_pretrained(MODEL_ID, revision=REV)
model = Qwen2AudioForConditionalGeneration.from_pretrained(MODEL_ID, revision=REV, torch_dtype=torch.float32,
                                                           attn_implementation="eager").to("cuda").eval()
model.audio_tower.config._attn_implementation = "eager"
model.language_model.config._attn_implementation = "eager"
print("so tham so: %.0f M" % (sum(p.numel() for p in model.parameters()) / 1e6))
print("audio", model.config.audio_config.encoder_layers, "lop | LLM", model.config.text_config.num_hidden_layers, "lop, hidden",
      model.config.text_config.hidden_size, "| tie:", model.config.text_config.tie_word_embeddings)
IM_END_ID = processor.tokenizer.convert_tokens_to_ids("<|im_end|>")
AUDIO_ID = getattr(model.config, "audio_token_id", None) or model.config.audio_token_index
SR = processor.feature_extractor.sampling_rate
INSTR = "You are a virtual assistant, answering questions and providing information in a clear and polite manner."
PROMPT = ("<|audio_bos|><|AUDIO|><|audio_eos|><|im_start|>user\nAudio content provided. " + INSTR + "<|im_end|>\n<|im_start|>assistant\n")

def make_inputs(path):
    audio, _ = librosa.load(path, sr=SR)
    r = processor(text=PROMPT, audio=audio, return_tensors="pt", sampling_rate=SR, padding=True)
    return {k: v.to("cuda") for k, v in r.items()}

GPU dang giu truoc khi nap: 0.00 GB (phai gan 0)


processor_config.json:   0%|          | 0.00/153 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/746 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/359 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

... (8 outputs left out)

model-00001-of-00002.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

so tham so: 1276 M
audio 32 lop | LLM 24 lop, hidden 896 | tie: False


In [6]:
# ===== 4. Tham chiếu FP: sinh greedy giống test_dialogue_inthewild.py =====
REF = []
for i, p in enumerate(WAVS):
    inp = make_inputs(p)
    t0 = time.time()
    with torch.no_grad():
        g = model.generate(**inp, max_new_tokens=MAX_NEW, do_sample=False, repetition_penalty=1.1, eos_token_id=IM_END_ID,
                           pad_token_id=processor.tokenizer.pad_token_id, use_cache=True)
    gen = g[0, inp["input_ids"].shape[1]:]
    text = processor.batch_decode(gen[None], skip_special_tokens=True)[0].strip()
    REF.append(dict(path=p, inp=inp, gen=gen, text=text))
    print(f"[{i+1:2d}] audio_tokens={int((inp['input_ids']==AUDIO_ID).sum())} prompt={inp['input_ids'].shape[1]} "
          f"sinh={len(gen)} ({time.time()-t0:.1f}s)\n     Q: {QUESTIONS[i] if i < len(QUESTIONS) else ''}\n     A: {text}")

[ 1] audio_tokens=172 prompt=204 sinh=91 (5.2s)
     Q: If you had to live in a video game universe for a year, which one would you choose?
     A: That's an interesting question! Personally, I think the universe would be incredibly peaceful if it were a good old-fashioned video game world. There wouldn't be any conflict or violence between players. On the other hand, some people might find that a more realistic or fair scenario could have more consequences. It really depends on what kind of balance you're looking for in such a setting. What about you? Which universe do you imagine being the most balanced?
[ 2] audio_tokens=186 prompt=218 sinh=90 (3.7s)
     Q: Would you rather be able to speak every language fluently or play every musical instrument perfectly?
     A: That's an interesting question! Personally, I think it would be fun to have the ability to speak every language fluently, as that could make travel even more exciting. On the other hand, playing all musical instruments p

In [7]:
# ===== 5. Boc model bang QLinear/QConv1d; kiem tra che do 'off' KHONG doi ket qua =====
import qint8_sim as sim
lf0, n0, full0 = sim.forward_logits(model, REF[0]["inp"], REF[0]["gen"])      # truoc khi boc
sim.install(model)
sim.QS.mode = "off"
lf1, _, _ = sim.forward_logits(model, REF[0]["inp"], REF[0]["gen"])
print("sau khi boc, che do off: max|diff| logit =", float((lf0 - lf1).abs().max()), "(phai = 0)")
assert float((lf0 - lf1).abs().max()) == 0.0
print("so GEMM duoc boc:", sum(isinstance(m, sim.QGemm) for m in model.modules()))

sau khi boc, che do off: max|diff| logit = 0.0 (phai = 0)
so GEMM duoc boc: 364


In [8]:
# ===== 5b. Kiem tra co che fallback: scale cuc nho => MOI hang bao hoa => moi thu chay FP => logit phai == FP =====
sim.QS.mode = "calib"; sim.QS.row_fb = False
sim.set_valid(REF[0]["inp"]["feature_attention_mask"]); sim.start_calib()
sim.forward_logits(model, REF[0]["inp"], REF[0]["gen"])
tiny = {k: 1e-6 for k in sim.build_scales([sim.stats_of_clip()]).keys()}
sim.QS.scales, sim.QS.fp_layers = tiny, {"conv1", "conv2"}          # conv khong co fallback
sim.QS.score_mode, sim.QS.lm_out, sim.QS.row_fb, sim.QS.fb_w = "int32", "int32", True, "fp"
sim.QS.mode = "quant"; sim.reset_fb()
lq, _, _ = sim.forward_logits(model, REF[0]["inp"], REF[0]["gen"])
sim.QS.mode, sim.QS.row_fb, sim.QS.fb_w, sim.QS.fp_layers = "off", False, "int8", set()
d = float((lq.cpu() - lf0.cpu()).abs().max())
print("fallback toan bo (trong so FP): max|diff| logit so voi FP =", d, "(phai rat nho, ~1e-3)")
assert d < 0.05

fallback toan bo (trong so FP): max|diff| logit so voi FP = 4.57763671875e-05 (phai rat nho, ~1e-3)


In [9]:
# ===== 6. Kiem tra so hoc so nguyen (torch) so voi ban Python thuan =====
import math, random
def py_qparam(r):
    shift = min(62, int(math.floor(math.log2((2**31 - 1) / r))))
    mult = int(round(math.ldexp(r, shift)))
    while mult > 2**31 - 1 and shift > 0:
        shift -= 1; mult = int(round(math.ldexp(r, shift)))
    return mult, shift
random.seed(1)
reals = [10 ** random.uniform(-7, 0.5) for _ in range(2000)]
m_t, s_t = sim.qparams(torch.tensor(reals, dtype=torch.float64, device="cuda"))
bad = sum(1 for r, mt, st in zip(reals, m_t.tolist(), s_t.tolist()) if py_qparam(r) != (mt, st))
print("qparams khac Python:", bad, "/", len(reals))
acc = torch.randint(-2**26, 2**26, (64, 2000), device="cuda").double()
bias = torch.randint(-2**20, 2**20, (2000,), device="cuda")
v_t = sim.requant(acc, m_t, s_t, bias).cpu().tolist()
accl, biasl = acc.long().cpu().tolist(), bias.cpu().tolist()
ms, ss = m_t.tolist(), s_t.tolist()
bad = 0
for i in range(64):
    for j in range(0, 2000, 7):
        v = ((accl[i][j] + biasl[j]) * ms[j] + ((1 << (ss[j] - 1)) if ss[j] > 0 else 0)) >> ss[j]
        bad += int(max(-128, min(127, v)) != v_t[i][j])
print("requant khac Python:", bad)
x8 = torch.randint(-128, 128, (50, 5120), device="cuda").float()
W8 = torch.randint(-128, 128, (300, 5120), device="cuda", dtype=torch.int8)
ref = (x8.long().cpu().numpy() @ W8.long().cpu().numpy().T)
print("int_mm khac numpy int64:", int((sim.int_mm(x8, W8).long().cpu().numpy() != ref).sum()))

qparams khac Python: 0 / 2000
requant khac Python: 0
int_mm khac numpy int64: 0


## 2b. Module lam muot `qsmooth.py`

In [10]:
%%writefile qsmooth.py
"""
qsmooth.py - Lam muot kenh (SmoothQuant) NGOAI TUYEN, tuong duong FP, cho bo mo phong qint8_sim.

Y tuong: mot vai kenh dau vao co bien do lon bat thuong (vd. dau vao down_proj) khien scale per-tensor qua lon. Chia kenh j cua
dau vao cho s_j va nhan cot j cua trong so voi s_j thi ket qua FP khong doi. Phan chia duoc GOP vao tham so lop truoc (khong them
phep tinh nao tren board):
  down_proj : PS nhan dau vao (silu(gate)*up) voi 1/s theo kenh (gop vao buoc nhan silu*up, them 1 phep nhan/phan tu), nhan cot
              down_proj voi s. KHONG gop vao up_proj: dau ra INT8 per-tensor cua up_proj se bi lech phan bo kenh (da do: top-1 cua up_proj
              tu 94% xuong 82%).
  gate, up  : chia trong so RMSNorm post_attention_layernorm cho s
  q, k, v   : chia trong so RMSNorm input_layernorm cho s
  q/k       : nhan hang q_proj voi t, chia hang k_proj cho t (cap kenh RoPE (i, i+d/2) dung chung t; GQA: t theo nhom kv)
  encoder   : LayerNorm cho qkv va fc1; v_proj (hang) cho out_proj
Moi thay doi deu hoan nguyen duoc bang restore().
"""
import torch

_ORIG = {}
_TOUCH = {}
_HOOKS = []


def _save(t):
    if id(t) not in _ORIG:
        _ORIG[id(t)] = (t, t.detach().cpu().clone())


def _touch(m):
    _TOUCH[id(m)] = m


@torch.no_grad()
def _refresh(m):
    """Luong tu lai trong so INT8 per-channel cua mot QLinear sau khi trong so FP bi doi."""
    W = m.lin.weight.detach()
    N = W.shape[0]
    for n0 in range(0, N, 8192):
        w = W[n0:n0 + 8192].double()
        amax = w.abs().amax(dim=1)
        s = torch.where(amax > 0, amax / 127.0, torch.ones_like(amax))
        m.W8[n0:n0 + 8192] = torch.clamp(torch.round(w / s[:, None]), -128, 127).to(torch.int8)
        m.sw[n0:n0 + 8192] = s
    m.bias = None if m.lin.bias is None else m.lin.bias.detach().double()
    m._cache = None


def _mul_cols(m, s):                 # W[:, j] *= s[j]
    _save(m.lin.weight); _touch(m)
    m.lin.weight.data *= s.to(m.lin.weight.device, m.lin.weight.dtype)[None, :]


def _scale_rows(m, s, inverse):      # W[i, :] (va bias[i]) *= s[i]  hoac /= s[i]
    _save(m.lin.weight); _touch(m)
    sd = s.to(m.lin.weight.device, m.lin.weight.dtype)
    f = (1.0 / sd) if inverse else sd
    m.lin.weight.data *= f[:, None]
    if m.lin.bias is not None:
        _save(m.lin.bias)
        m.lin.bias.data *= f


def _div_vec(t, s):
    _save(t)
    t.data /= s.to(t.device, t.dtype)


def _smooth(X, Wcol, a):
    X, Wcol = X.to(Wcol.device).float().clamp_min(1e-5), Wcol.float().clamp_min(1e-5)
    s = X.pow(a) / Wcol.pow(1.0 - a)
    return s.clamp(1e-3, 1e3)


@torch.no_grad()
def restore(model):
    for h in _HOOKS:
        h.remove()
    _HOOKS.clear()
    for t, c in _ORIG.values():
        t.data.copy_(c.to(t.device))
    _ORIG.clear()
    for m in list(_TOUCH.values()):
        _refresh(m)
    _TOUCH.clear()


@torch.no_grad()
def collect_chmax(model, sim, run_fn, c=5.0):
    """Max |x| theo tung kenh dau vao cua moi QLinear (bo hang ngoai lai), va max cap-kenh cua q/k (truoc RoPE).
    run_fn(): ham chay cac clip o che do 'off' (FP)."""
    ch = {}
    hooks = []
    keep_of = lambda r: (r <= c * r.median()) if float(r.median()) > 0 else torch.ones_like(r, dtype=torch.bool)

    def upd(name, v):
        ch[name] = torch.maximum(ch[name], v) if name in ch else v

    def valid(mod, x):
        n = sim.QS.valid.get(mod.valid_key) if mod.valid_key else None
        return x[:, :n] if n else x

    def pre(mod, args):
        x = valid(mod, args[0].detach()).float()
        x2 = x.reshape(-1, x.shape[-1]).abs()
        r = x2.amax(1)
        upd(mod.name, x2[keep_of(r)].amax(0).cpu())

    def post_qk(kind, nh, d):
        def f(mod, args, out):
            y = valid(mod, out.detach()).float().reshape(-1, nh, d)
            r = y.abs().amax(dim=(1, 2))
            y = y[keep_of(r)]
            mag = torch.sqrt(y[..., : d // 2] ** 2 + y[..., d // 2:] ** 2).amax(0)        # [nh, d/2]
            upd(mod.name + ".pair", mag.cpu())
        return f

    cfg = model.language_model.config
    d = cfg.hidden_size // cfg.num_attention_heads
    for m in model.modules():
        if isinstance(m, sim.QLinear):
            hooks.append(m.register_forward_pre_hook(pre))
            if m.name.startswith("llm.") and m.name.endswith(".q_proj"):
                hooks.append(m.register_forward_hook(post_qk("q", cfg.num_attention_heads, d)))
            if m.name.startswith("llm.") and m.name.endswith(".k_proj"):
                hooks.append(m.register_forward_hook(post_qk("k", cfg.num_key_value_heads, d)))
    sim.QS.mode = "off"
    try:
        run_fn()
    finally:
        for h in hooks:
            h.remove()
    return ch


@torch.no_grad()
def apply(model, sim, CH, a_down=None, a_gu=None, a_qkv=None, a_qk=None, a_enc=None):
    """Ap dung lam muot. Moi tham so a_* la alpha (None = tat). Tra ve dict thong ke so s."""
    info = {}
    cfg = model.language_model.config
    Hq, Hkv = cfg.num_attention_heads, cfg.num_key_value_heads
    d = cfg.hidden_size // Hq
    rep = Hq // Hkv
    for j, L in enumerate(model.language_model.model.layers):
        a, m = L.self_attn, L.mlp
        if a_down is not None:
            s = _smooth(CH[f"llm.{j}.down_proj"], m.down_proj.lin.weight.abs().amax(0), a_down)
            _mul_cols(m.down_proj, s)
            inv = (1.0 / s).to(m.down_proj.lin.weight.device)
            _HOOKS.append(m.down_proj.register_forward_pre_hook(
                lambda mod, args, inv=inv: (args[0] * inv.to(args[0].dtype),) + tuple(args[1:])))
            info[f"llm.{j}.down"] = (float(s.min()), float(s.max()))
        if a_gu is not None:
            Wc = torch.maximum(m.gate_proj.lin.weight.abs().amax(0), m.up_proj.lin.weight.abs().amax(0))
            s = _smooth(CH[f"llm.{j}.gate_proj"], Wc, a_gu)
            _div_vec(L.post_attention_layernorm.weight, s)
            _mul_cols(m.gate_proj, s); _mul_cols(m.up_proj, s)
            info[f"llm.{j}.gu"] = (float(s.min()), float(s.max()))
        if a_qkv is not None:
            Wc = torch.maximum(torch.maximum(a.q_proj.lin.weight.abs().amax(0), a.k_proj.lin.weight.abs().amax(0)),
                               a.v_proj.lin.weight.abs().amax(0))
            s = _smooth(CH[f"llm.{j}.q_proj"], Wc, a_qkv)
            _div_vec(L.input_layernorm.weight, s)
            for p in (a.q_proj, a.k_proj, a.v_proj):
                _mul_cols(p, s)
            info[f"llm.{j}.qkv"] = (float(s.min()), float(s.max()))
        if a_qk is not None:
            qm = CH[f"llm.{j}.q_proj.pair"].reshape(Hkv, rep, d // 2).amax(1)           # [Hkv, d/2]
            km = CH[f"llm.{j}.k_proj.pair"]                                              # [Hkv, d/2]
            t = (km.clamp_min(1e-5) / qm.clamp_min(1e-5)).pow(0.5 * a_qk).clamp(1e-2, 1e2)
            t = torch.cat([t, t], dim=1)                                                 # [Hkv, d]
            tq = t.repeat_interleave(rep, 0).reshape(-1)
            _scale_rows(a.q_proj, tq, inverse=False)
            _scale_rows(a.k_proj, t.reshape(-1), inverse=True)
            info[f"llm.{j}.qk"] = (float(t.min()), float(t.max()))
    if a_enc is not None:
        for i, L in enumerate(model.audio_tower.layers):
            a = L.self_attn
            Wc = torch.maximum(torch.maximum(a.q_proj.lin.weight.abs().amax(0), a.k_proj.lin.weight.abs().amax(0)),
                               a.v_proj.lin.weight.abs().amax(0))
            s = _smooth(CH[f"enc.{i}.q_proj"], Wc, a_enc)
            _div_vec(L.self_attn_layer_norm.weight, s); _div_vec(L.self_attn_layer_norm.bias, s)
            for p in (a.q_proj, a.k_proj, a.v_proj):
                _mul_cols(p, s)
            s = _smooth(CH[f"enc.{i}.fc1"], L.fc1.lin.weight.abs().amax(0), a_enc)
            _div_vec(L.final_layer_norm.weight, s); _div_vec(L.final_layer_norm.bias, s)
            _mul_cols(L.fc1, s)
            s = _smooth(CH[f"enc.{i}.out_proj"], a.out_proj.lin.weight.abs().amax(0), a_enc)
            _scale_rows(a.v_proj, s, inverse=True)
            _mul_cols(a.out_proj, s)
            info[f"enc.{i}"] = (float(s.min()), float(s.max()))
    for m in list(_TOUCH.values()):
        _refresh(m)
    return info

Writing qsmooth.py


In [11]:
# ===== 7. Chuan FP32 + moc bf16/fp16: teacher-forced tren cac clip hoi thoai (TRUOC khi lam muot, che do 'off' = FP goc) =====
import numpy as np, contextlib, re, math
torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False      # mo phong INT8 chinh xac can fp32 that, khong TF32
DEV = next(model.parameters()).device.type

def pos_stats(lf, lx, full, n0, penalty=1.1):
    # So sanh logit X voi FP32 tung vi tri (cung phat lap lai 1,1 nhu luc sinh).
    # gap = logp_FP(token FP chon) - logp_FP(token X chon) >= 0 : FP "tiec" bao nhieu nat khi X chon khac; ~0 nghia la FP cung phan van.
    dev = full.device
    lf = lf.float().to(dev); lx = lx.float().to(dev)
    flips, gaps, ranks, margins = [], [], [], []
    for t in range(lf.shape[0]):
        hist = full[:n0 + t]
        a = sim.apply_penalty(lf[t].clone(), hist, penalty)
        b = sim.apply_penalty(lx[t].clone(), hist, penalty)
        lp = torch.log_softmax(a, -1)
        ta, tb = int(a.argmax()), int(b.argmax())
        top2 = lp.topk(2).values
        flips.append(ta != tb); gaps.append(float(lp[ta] - lp[tb])); margins.append(float(top2[0] - top2[1]))
        ranks.append(int((lp > lp[tb]).sum()) + 1)
    return dict(flip=flips, gap=gaps, rank=ranks, margin=margins)

def merge(parts):
    out = {}
    for p in parts:
        for k, v in p.items(): out.setdefault(k, []).extend(v)
    return out

def tie_summary(P):
    fl, gp, rk = np.array(P["flip"]), np.array(P["gap"]), np.array(P["rank"])
    n, nf = len(fl), int(np.sum(fl))
    g = gp[fl]
    frac = lambda m: float(np.mean(m)) if nf else 0.0
    return dict(n=n, top1=1 - nf / n, flips=nf,
                near01=frac(g < 0.1), near05=frac(g < 0.5), lt1=frac(g < 1.0), lt2=frac(g < 2.0),
                severe_per1k=1000.0 * float(np.sum(g >= 2.0)) / n,
                regret=float(np.mean(gp)), rank2=frac(rk[fl] <= 2))

# --- FP32 tham chieu (luu logit CPU cho moi clip de so voi INT8 sau)
sim.QS.mode = "off"
for r in REF:
    lf, n0, full = sim.forward_logits(model, r["inp"], r["gen"])
    r["lf"], r["n0"], r["full"] = lf.cpu(), n0, full
print("FP32: ", len(REF), "clip, ", sum(r["lf"].shape[0] for r in REF), "vi tri token")

# --- bf16 / fp16 (autocast) so voi FP32
BASE_STATS, BASE_DTYPE = {}, {"bf16": torch.bfloat16, "fp16": torch.float16}
for tag, dt in BASE_DTYPE.items():
    try:
        parts, bad, t0 = [], 0, time.time()
        for r in REF:
            with torch.no_grad(), torch.autocast(DEV, dtype=dt):
                lx, _, _ = sim.forward_logits(model, r["inp"], r["gen"])
            if not bool(torch.isfinite(lx).all()): bad += 1; continue
            parts.append(pos_stats(r["lf"], lx, r["full"], r["n0"]))
        if bad: print(f"{tag}: {bad} clip bi NaN/Inf -> bo moc nay")
        elif parts: BASE_STATS[tag] = merge(parts); print(f"{tag}: xong [{time.time()-t0:.0f}s]")
    except Exception as e:
        print(f"{tag}: khong chay duoc ({type(e).__name__}: {str(e)[:100]})")
assert BASE_STATS, "khong co moc bf16/fp16 nao chay duoc"
BF_TAG = "bf16" if "bf16" in BASE_STATS else "fp16"
print("moc dung cho sinh cau tra loi:", BF_TAG)

FP32:  50 clip,  3395 vi tri token
bf16: xong [70s]
fp16: xong [25s]
moc dung cho sinh cau tra loi: bf16


In [12]:
QUIZ_RAW = [["What is the capital city of France?", ["\\bParis\\b"]], ["What is the capital of Japan?", ["\\bTokyo\\b"]], ["What is the longest river in Africa?", ["\\bNile\\b"]], ["Which is the largest country in the world by area?", ["\\bRussia\\b"]], ["What is the capital city of Australia?", ["\\bCanberra\\b"]], ["Which desert is the largest hot desert in the world?", ["\\bSahara\\b"]], ["On which continent is Brazil located?", ["South America"]], ["What is the capital of Canada?", ["\\bOttawa\\b"]], ["Which river flows through Paris?", ["\\bSeine\\b"]], ["What is the smallest country in the world?", ["\\bVatican\\b"]], ["Which ocean lies between Africa and Australia?", ["\\bIndian\\b"]], ["What is the capital city of Italy?", ["\\bRome\\b"]], ["In which country is the city of Kyoto?", ["\\bJapan\\b"]], ["What is the capital of Egypt?", ["\\bCairo\\b"]], ["Which mountain range separates Europe from Asia?", ["\\bUral"]], ["What is the capital city of Thailand?", ["\\bBangkok\\b"]], ["What is the longest river in South America?", ["\\bAmazon\\b"]], ["Which sea is located between Europe and Africa?", ["\\bMediterranean\\b"]], ["What is the capital of Germany?", ["\\bBerlin\\b"]], ["Which country is shaped like a boot?", ["\\bItaly\\b"]], ["What is the largest island in the world?", ["\\bGreenland\\b"]], ["What is the capital city of Spain?", ["\\bMadrid\\b"]], ["Which river flows through Egypt?", ["\\bNile\\b"]], ["What is the capital of South Korea?", ["\\bSeoul\\b"]], ["Which city is known as the Big Apple?", ["New York"]], ["What is the currency of Vietnam?", ["d[oồ]ng", "\\bVND\\b"]], ["What is the largest city in Vietnam by population?", ["Ho Chi Minh", "Saigon", "Sài Gòn"]], ["Which river flows through the south of Vietnam and is known as the Nine Dragon river?", ["\\bMekong\\b"]], ["Who was the first president of the Democratic Republic of Vietnam?", ["H[oồ] Ch[ií] Minh"]], ["What is the traditional Vietnamese dress called?", ["[aá]o d[aà]i"]], ["What do Vietnamese people call the Lunar New Year festival?", ["\\bT[eế]t\\b"]], ["Which city in central Vietnam is famous for its ancient town with yellow lanterns?", ["H[oộ]i An"]], ["What is the name of the highest mountain in Vietnam?", ["Fansipan", "Phan Xi P[aă]ng", "Fan Si Pan"]], ["Which city in the highlands of Vietnam is known as the city of eternal spring?", ["Da ?Lat", "Đ[aà] L[aạ]t"]], ["Which Vietnamese city was the imperial capital under the Nguyen dynasty?", ["\\bHu[eế]\\b"]], ["Which famous Vietnamese drink is made by whisking egg yolk with coffee?", ["egg coffee", "c[aà] ph[eê] tr[uứ]ng", "ca phe trung"]], ["Which landlocked country lies to the west of Vietnam?", ["\\bLaos\\b"]], ["What is the chemical symbol for sodium?", ["\\bNa\\b"]], ["What is the chemical formula of water?", ["H2O", "H₂O", "H\\s*2\\s*O"]], ["How many planets are in our solar system?", ["\\beight\\b", "\\b8\\b"]], ["Which planet is closest to the Sun?", ["\\bMercury\\b"]], ["What is the largest planet in our solar system?", ["\\bJupiter\\b"]], ["What is the hardest natural substance on Earth?", ["\\bdiamond"]], ["What gas do humans need to breathe to survive?", ["\\boxygen\\b"]], ["What is the center of an atom called?", ["\\bnucleus\\b"]], ["What is the speed of light approximately in kilometers per second?", ["299[, ]?792", "300[, ]?000", "3\\s*[x×]\\s*10"]], ["What force keeps us on the ground?", ["\\bgravity\\b"]], ["What is the chemical symbol for iron?", ["\\bFe\\b"]], ["What is the process by which water changes from liquid to gas?", ["evaporat", "vapori[sz]"]], ["Which planet is known for its prominent rings?", ["\\bSaturn\\b"]], ["What is the main gas in the Earth's atmosphere?", ["\\bnitrogen\\b"]], ["What is the freezing point of water in degrees Celsius?", ["\\b0\\b", "\\bzero\\b"]], ["Which organ do we use for breathing?", ["\\blungs?\\b"]], ["What is the name of the galaxy that contains our solar system?", ["Milky Way"]], ["How many bones are in the adult human body?", ["\\b206\\b"]], ["What is the closest star to Earth?", ["\\bSun\\b"]], ["What type of animal is a dolphin?", ["\\bmammal"]], ["What is the study of living things called?", ["\\bbiology\\b"]], ["What is the powerhouse of the cell?", ["mitochondri"]], ["What is the chemical symbol for silver?", ["\\bAg\\b"]], ["Which vitamin do we get from sunlight?", ["vitamin\\s*D\\b"]], ["What is the largest organ of the human body?", ["\\bskin\\b"]], ["What is the unit of electrical resistance?", ["\\bohms?\\b"]], ["Which blood cells carry oxygen?", ["red blood", "erythrocyte", "h(a)?emoglobin"]], ["What is the boiling point of water in degrees Fahrenheit?", ["\\b212\\b"]], ["How many hours are in a day?", ["\\b24\\b", "twenty[- ]four"]], ["What is the largest mammal in the world?", ["blue whale"]], ["Who painted the Mona Lisa?", ["Leonardo", "da Vinci"]], ["Who was the first president of the United States?", ["Washington"]], ["In which year did World War Two end?", ["\\b1945\\b"]], ["Who discovered penicillin?", ["Fleming"]], ["Who wrote the novel Pride and Prejudice?", ["Austen"]], ["Which ancient civilization built the pyramids of Giza?", ["Egypt"]], ["Who was the first person to walk on the Moon?", ["Armstrong"]], ["In which year did the Titanic sink?", ["\\b1912\\b"]], ["Who developed the theory of relativity?", ["Einstein"]], ["Which city hosted the 2008 Summer Olympics?", ["Beijing"]], ["Who wrote the play Hamlet?", ["Shakespeare"]], ["What is the name of the famous clock tower in London?", ["Big Ben", "Elizabeth Tower"]], ["Who composed the Four Seasons?", ["Vivaldi"]], ["Which ancient wonder was located in Alexandria?", ["Lighthouse", "Pharos"]], ["Who invented the telephone?", ["\\bBell\\b"]], ["In which year did the Berlin Wall fall?", ["\\b1989\\b"]], ["Who was the first woman to win a Nobel Prize?", ["Curie"]], ["Which empire was ruled by Julius Caesar?", ["Roman"]], ["Who painted the ceiling of the Sistine Chapel?", ["Michelangelo"]], ["Who wrote the Odyssey?", ["Homer"]], ["Which country gave the Statue of Liberty to the United States?", ["France"]], ["Who was the British prime minister during most of World War Two?", ["Churchill"]], ["In which country did the Olympic Games originate?", ["Greece"]], ["Who wrote the novel Don Quixote?", ["Cervantes"]], ["What is the name of the longest wall in the world, built in China?", ["Great Wall"]], ["In computer science, what does the acronym RAM stand for?", ["random[- ]access memory"]], ["What does the acronym HTML stand for?", ["hyper\\s*text markup"]], ["Which company created the iPhone?", ["\\bApple\\b"]], ["What does the acronym USB stand for?", ["universal serial bus"]], ["Who founded Microsoft together with Paul Allen?", ["Gates"]], ["Which programming language is named after a type of snake?", ["\\bPython\\b"]], ["What does the acronym GPS stand for?", ["global positioning system"]], ["How many bits are in a byte?", ["\\beight\\b", "\\b8\\b"]], ["What does the acronym AI stand for?", ["artificial intelligence"]], ["What is the square root of sixty four?", ["\\beight\\b", "\\b8\\b"]], ["What is seven multiplied by eight?", ["\\b56\\b", "fifty[- ]six"]], ["What is the value of pi to two decimal places?", ["3\\.14"]], ["How many sides does a hexagon have?", ["\\bsix\\b", "\\b6\\b"]], ["What is the sum of the angles in a triangle in degrees?", ["\\b180\\b", "one hundred (and )?eighty"]], ["What is fifteen percent of two hundred?", ["\\b30\\b", "thirty"]], ["How many degrees are in a right angle?", ["\\b90\\b", "ninety"]], ["What is the next prime number after seven?", ["\\b11\\b", "eleven"]], ["What is the square of twelve?", ["\\b144\\b", "one hundred (and )?forty[- ]four"]], ["What is one hundred divided by four?", ["\\b25\\b", "twenty[- ]five"]], ["How many minutes are in an hour?", ["\\b60\\b", "sixty"]], ["What does the acronym URL stand for?", ["uniform resource locator"]], ["Which company owns the video platform YouTube?", ["Google", "Alphabet"]], ["What is the name of the operating system developed by Apple for the Mac?", ["macOS", "Mac OS", "OS X"]], ["What does the acronym PDF stand for?", ["portable document format"]], ["Which company makes the Windows operating system?", ["Microsoft"]], ["What is the main language used to style web pages?", ["\\bCSS\\b", "cascading style"]], ["What is the name of the social network founded by Mark Zuckerberg?", ["Facebook", "\\bMeta\\b"]], ["What does the acronym SQL stand for?", ["structured query language"]], ["What is ten squared?", ["\\b100\\b", "one hundred"]], ["What color do you get by mixing blue and yellow?", ["\\bgreen\\b"]], ["How many days are there in a leap year?", ["\\b366\\b"]], ["What is the opposite of the word hot?", ["\\bcold\\b"]], ["Which animal is known as the king of the jungle?", ["\\blion\\b"]], ["How many strings does a standard guitar have?", ["\\bsix\\b", "\\b6\\b"]], ["What is the largest bird in the world?", ["\\bostrich"]], ["Which fruit is traditionally said to keep the doctor away?", ["\\bapple"]], ["What is the main ingredient in guacamole?", ["\\bavocado"]], ["Which sport is played at Wimbledon?", ["\\btennis\\b"]], ["How many players from one team are on the field in a soccer match?", ["\\b11\\b", "eleven"]], ["Which insect produces honey?", ["\\bbees?\\b"]], ["How many months are there in a year?", ["\\b12\\b", "twelve"]], ["What is the main ingredient of bread?", ["\\bflour\\b"]], ["Which country is famous for inventing sushi?", ["\\bJapan\\b"]], ["What is a baby cat called?", ["\\bkitten"]], ["What is the day after Monday?", ["\\bTuesday\\b"]], ["How many colors are there in a rainbow?", ["\\bseven\\b", "\\b7\\b"]], ["Which animal is the tallest in the world?", ["\\bgiraffe"]], ["In which sport do players hit a shuttlecock?", ["\\bbadminton\\b"]], ["What is the main language spoken in Brazil?", ["\\bPortuguese\\b"]], ["What is the currency of the United Kingdom?", ["\\bpound", "sterling"]], ["How many letters are in the English alphabet?", ["\\b26\\b", "twenty[- ]six"]], ["Which instrument is played with a bow, has four strings and is smaller than a cello?", ["\\bviolin\\b"]], ["What is the largest continent by land area?", ["\\bAsia\\b"]], ["What is a group of lions called?", ["\\bpride\\b"]], ["What is the main language spoken in Spain?", ["\\bSpanish\\b"]], ["What is the Italian dish made of flat dough topped with cheese and tomato?", ["\\bpizza\\b"]], ["Which fruit is dried to make raisins?", ["\\bgrapes?\\b"]], ["Which animal appears with the emu on the coat of arms of Australia?", ["\\bkangaroo"]]]
CATS = [["dia ly", 25], ["Viet Nam", 12], ["khoa hoc", 30], ["lich su", 25], ["may tinh/toan", 29], ["kien thuc chung", 29]]
# ===== 8. Bo cau hoi co dap an: chon deu theo QUIZ_N de phu moi nhom; sinh wav bang edge-tts (can Internet) =====
SEL = sorted(set(int(round(x)) for x in np.linspace(0, len(QUIZ_RAW) - 1, min(QUIZ_N, len(QUIZ_RAW)))))
QSEL = [(k + 1, QUIZ_RAW[k][0], QUIZ_RAW[k][1]) for k in SEL]
QDIR = f"{GEN_DIR}/quiz"; os.makedirs(QDIR, exist_ok=True)
qpath = lambda k: f"{QDIR}/quiz_{k:03d}.wav"
todo = [(qpath(k), q, k) for k, q, _ in QSEL if not os.path.exists(qpath(k))]
if todo:
    print(f"Dang tao {len(todo)} file wav cho bo cau hoi (edge-tts, can Internet)...")
    t0 = time.time(); tts_all(todo); print(f"xong [{time.time()-t0:.0f}s]")
QUIZ = [dict(k=k, q=q, pats=p, wav=qpath(k)) for k, q, p in QSEL if os.path.exists(qpath(k))]
def cat_of(k):
    acc = 0
    for name, cnt in CATS:
        acc += cnt
        if k <= acc: return name
    return CATS[-1][0]
print(len(QUIZ), "cau co wav /", len(QSEL), "yeu cau |", {c: sum(1 for q in QUIZ if cat_of(q["k"]) == c) for c, _ in CATS})
if len(QUIZ) < len(QSEL): print("CANH BAO: thieu", len(QSEL) - len(QUIZ), "cau (edge-tts loi?)")
assert len(QUIZ) >= 5

Dang tao 50 file wav cho bo cau hoi (edge-tts, can Internet)...
xong [23s]
50 cau co wav / 50 yeu cau | {'dia ly': 9, 'Viet Nam': 4, 'khoa hoc': 9, 'lich su': 9, 'may tinh/toan': 9, 'kien thuc chung': 10}


In [13]:
# ===== 9. Cham diem + kiem tra bo cham =====
def sentences(t): return [s for s in re.split(r"(?<=[.!?])\s+", t.strip()) if s]
def grade(text, pats):
    # s1: mau dung co trong CAU DAU (muc chinh, nghiem nhat); s2: trong 2 cau dau; any: o bat ky dau (can tren, co the tinh nham khi
    # cau tra loi chi nhac den dap an dung o ve khac, vd. "surpassing the Nile"). Khong gioi han so ky tu.
    rx = [re.compile(p, re.I) for p in pats]
    ss = sentences(text)
    h1, h2 = " ".join(ss[:1]), " ".join(ss[:2])
    return dict(s1=any(r.search(h1) for r in rx), s2=any(r.search(h2) for r in rx), any=any(r.search(text) for r in rx))

assert grade("The Red Planet is Mars.", [r"\bMars\b"])["s1"]
g = grade("It is Jupiter. It is very large. Mars is smaller.", [r"\bMars\b"])
assert (not g["s1"]) and (not g["s2"]) and g["any"]
g = grade("The longest river in Africa is the Congo River. It is large, surpassing the Nile.", [r"\bNile\b"])
assert (not g["s1"]) and g["s2"] and g["any"]                  # truong hop cua v11: tra loi SAI o cau dau nhung nhac 'Nile' o cau sau -> s1 sai, s2/any tinh nham
g = grade("The longest river in Africa is the Congo River. It is big. Rivers like the Nile are long.", [r"\bNile\b"])
assert (not g["s1"]) and (not g["s2"]) and g["any"]
g = grade("Okay. Paris is the capital. It is big.", [r"\bParis\b"])
assert (not g["s1"]) and g["s2"] and g["any"]
assert grade("Pi is approximately 3.14. Next sentence.", [r"3\.14"])["s1"]
assert not grade("The capital is Sydney.", [r"\bCanberra\b"])["any"]
assert grade("There are 8 bits in a byte.", [r"\beight\b", r"\b8\b"])["s1"]
print("bo cham diem: kiem tra don vi OK")

def mcnemar_p(b, c):
    n = b + c
    if n == 0: return 1.0
    k = min(b, c)
    return min(1.0, 2.0 * sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n)

bo cham diem: kiem tra don vi OK


In [14]:
# ===== 10. Sinh cau tra loi day du (dung o token ket thuc; KHONG cat ky tu) + chay FP32 va moc bf16/fp16 =====
QUIZ_RES = {}
def quiz_path(mode): return f"{OUT_DIR}/quiz_{mode}.json"
def fb_rate(excl=".attn"):
    rows = [(k, v) for k, v in sim.QS.fb.items() if not k.endswith(excl) and v[1] > 0]
    return 100.0 * sum(v[0] for _, v in rows) / max(sum(v[1] for _, v in rows), 1)
def quiz_run(mode):
    os.makedirs(OUT_DIR, exist_ok=True)
    done = json.load(open(quiz_path(mode))) if os.path.exists(quiz_path(mode)) else []
    ok = 0                                                 # chi giu phan da xong khop dung danh sach cau hoi hien tai
    while ok < min(len(done), len(QUIZ)) and done[ok].get("k") == QUIZ[ok]["k"]: ok += 1
    if ok < len(done): print(f"  [{mode}] bo {len(done) - ok} ket qua cu khong khop danh sach cau hoi")
    done = done[:ok]
    if ok: print(f"  [{mode}] tiep tuc tu cau {ok + 1} (da co {ok})")
    t00, k0 = time.time(), len(done)
    for k in range(len(done), len(QUIZ)):
        item = QUIZ[k]; inp = make_inputs(item["wav"])
        sim.set_valid(inp["feature_attention_mask"])
        if mode == "int8":
            sim.reset_fb(); sim.QS.mode, sim.QS.row_fb = "quant", True
        ctx = torch.autocast(DEV, dtype=BASE_DTYPE[BF_TAG]) if mode == "lowprec" else contextlib.nullcontext()
        try:
            with torch.no_grad(), ctx:
                g = model.generate(**inp, max_new_tokens=QUIZ_MAX_NEW, do_sample=False, repetition_penalty=1.1,
                                   eos_token_id=IM_END_ID, pad_token_id=processor.tokenizer.pad_token_id, use_cache=True)
        finally:
            sim.QS.mode, sim.QS.row_fb = "off", False
        gen = g[0, inp["input_ids"].shape[1]:]
        text = processor.batch_decode(gen[None], skip_special_tokens=True)[0].strip()
        capped = bool(len(gen) >= QUIZ_MAX_NEW and int(gen[-1]) != IM_END_ID)
        rec = dict(k=item["k"], text=text, n_tok=int(len(gen)), capped=capped, fb=None, fb_pre=None, fb_dec=None)
        if mode == "int8":
            sp = sim.fb_split()
            rec.update(fb=fb_rate(), fb_pre=sp.get("pre", (None, 0))[0], fb_dec=sp.get("dec", (None, 0))[0])
        done.append(rec)
        if (k + 1) % 5 == 0 or k + 1 == len(QUIZ):
            json.dump(done, open(quiz_path(mode), "w"), ensure_ascii=False)
            el = time.time() - t00; print(f"  [{mode}] {k+1}/{len(QUIZ)}  {el:.0f}s  (con ~{el/(k+1-k0)*(len(QUIZ)-k-1):.0f}s)  token={len(gen)}{' CAP' if capped else ''}")
    QUIZ_RES[mode] = done
    return done

sim.QS.mode = "off"
if "fp32" in QUIZ_MODES: print("== FP32 =="); quiz_run("fp32")
if "lowprec" in QUIZ_MODES: print(f"== {BF_TAG} (autocast) =="); quiz_run("lowprec")

== FP32 ==
  [fp32] 5/50  9s  (con ~78s)  token=10
  [fp32] 10/50  16s  (con ~65s)  token=25
  [fp32] 15/50  27s  (con ~62s)  token=65
  [fp32] 20/50  41s  (con ~62s)  token=90
  [fp32] 25/50  55s  (con ~55s)  token=97
  [fp32] 30/50  66s  (con ~44s)  token=40
  [fp32] 35/50  76s  (con ~33s)  token=16
  [fp32] 40/50  83s  (con ~21s)  token=23
  [fp32] 45/50  97s  (con ~11s)  token=29
  [fp32] 50/50  108s  (con ~0s)  token=18


In [15]:
# ===== 11. Lam muot + hieu chuan scale tren cac clip hoi thoai (giong v10: c=2, scale theo kenh) =====
import qsmooth as qs
STATS = None
def collect(c):
    sim.QS.fp_layers = set()
    sim.QS.mode = "calib"; sim.QS.row_fb = True; sim.QS.row_c = c; sim.QS.out_pc = True
    out = []
    for r in REF:
        sim.set_valid(r["inp"]["feature_attention_mask"]); sim.start_calib()
        sim.forward_logits(model, r["inp"], r["gen"])
        out.append(sim.stats_of_clip())
    sim.QS.mode = "off"; sim.QS.row_fb = False; sim.QS.out_pc = False
    return out
def _run_off():
    for r in REF:
        sim.set_valid(r["inp"]["feature_attention_mask"]); sim.forward_logits(model, r["inp"], r["gen"])
CH = qs.collect_chmax(model, sim, _run_off, c=2.0)
SM_KW = dict(a_down=0.5, a_gu=0.7, a_qkv=0.7, a_qk=1.0, a_enc=0.5)
qs.restore(model); SM_INFO = qs.apply(model, sim, CH, **SM_KW)
sim.QS.mode = "off"; sim.set_valid(REF[0]["inp"]["feature_attention_mask"])
lq0, _, _ = sim.forward_logits(model, REF[0]["inp"], REF[0]["gen"])
d = float((lq0.cpu() - REF[0]["lf"]).abs().max()); print("FP sau lam muot lech", d, "so voi FP goc (phai rat nho)"); assert d < 0.05
t0 = time.time(); STATS = collect(2.0); STATS_C = {2.0: STATS}; print(f"hieu chuan c=2 tren {len(STATS)} clip [{time.time()-t0:.0f}s]")
PC_MARGIN, ROW_C = 1.5, 2.0
def set_int8(scales):
    sim.QS.scales, sim.QS.fp_layers = scales, set()
    sim.QS.score_mode, sim.QS.lm_out, sim.QS.fb_w = "int32", "int32", "int8"
    sim.QS.int32_ops, sim.QS.attn_fp, sim.QS.out_pc, sim.QS.sat = (), (), True, "clip"
def unset_int8():
    sim.QS.mode, sim.QS.row_fb, sim.QS.out_pc, sim.QS.sat = "off", False, False, "edge"

FP sau lam muot lech 4.57763671875e-05 so voi FP goc (phai rat nho)
hieu chuan c=2 tren 50 clip [81s]


In [16]:
# ===== 12. Gioi han 3: cac lan lech co "vo hai" khong? INT8 (chia cheo 2 nhanh) so voi moc bf16/fp16 =====
n = len(REF); FOLDS = [list(range(0, n, 2)), list(range(1, n, 2))]
parts, agg_cmp = [], []
sim.reset_fb()
for test in FOLDS:
    train = [i for i in range(n) if i not in test]
    sc = sim.build_scales([STATS[i] for i in train], 100.0, 100.0, pc_margin=PC_MARGIN)
    set_int8(sc); sim.QS.mode, sim.QS.row_fb = "quant", True
    try:
        for i in test:
            r = REF[i]; sim.set_valid(r["inp"]["feature_attention_mask"])
            with torch.no_grad():
                lq, _, _ = sim.forward_logits(model, r["inp"], r["gen"])
            parts.append(pos_stats(r["lf"], lq, r["full"], r["n0"]))
            agg_cmp.append(sim.compare_logits(r["lf"], lq, r["full"], r["n0"]))
    finally:
        unset_int8()
CV_FB = fb_rate(); CV_SPLIT = sim.fb_split()
INT8_STATS = merge(parts)
print("kiem tra: top1 cua INT8 theo compare_logits =", round(sum(m["top1"] * m["n"] for m in agg_cmp) / sum(m["n"] for m in agg_cmp) * 100, 1), "% | hang FP (teacher-forced):", round(CV_FB, 2), "%")
TIE = {"INT8 P1.5": tie_summary(INT8_STATS)}
for tag, P in BASE_STATS.items(): TIE[tag + " (autocast)"] = tie_summary(P)
print("%-18s %7s %7s | %-34s | %9s %9s %8s" % ("", "top1", "lech", "ti le lech nho hon 0,1/0,5/1/2 nat", "nang/1000", "regret", "hang<=2"))
for k, s in TIE.items():
    print("%-18s %6.1f%% %7d | %5.0f%% %5.0f%% %5.0f%% %5.0f%%             | %9.1f %9.4f %7.0f%%" % (
        k, s["top1"] * 100, s["flips"], s["near01"] * 100, s["near05"] * 100, s["lt1"] * 100, s["lt2"] * 100, s["severe_per1k"], s["regret"], s["rank2"] * 100))
print("(lech 'nang' = FP gan nhu chac chan hon token INT8 chon, gap >= 2 nat; regret = nat/token FP mat khi lay token cua mo hinh kia)")

kiem tra: top1 cua INT8 theo compare_logits = 91.8 % | hang FP (teacher-forced): 1.32 %
                      top1    lech | ti le lech nho hon 0,1/0,5/1/2 nat | nang/1000    regret  hang<=2
INT8 P1.5            91.8%     280 |    25%    78%    96%   100%             |       0.3    0.0276      76%
bf16 (autocast)      98.0%      68 |    85%   100%   100%   100%             |       0.0    0.0011      93%
fp16 (autocast)      99.7%       9 |   100%   100%   100%   100%             |       0.0    0.0000      89%
(lech 'nang' = FP gan nhu chac chan hon token INT8 chon, gap >= 2 nat; regret = nat/token FP mat khi lay token cua mo hinh kia)


In [17]:
# ===== 12b. (chi cau hinh toi da) Quet them cac gia tri CHUA THU o v10: margin 1,75 / 2,0, c=1,5, INT32 moi GEMM voi c=2 =====
# Cau tra loi (cell 13) van dung P1.5 co dinh (da chot truoc), de ket qua cau hoi khong bi chon theo ket qua quet.
SWEEP = []
if RUN_SWEEP:
    t0 = time.time(); STATS_C[1.5] = collect(1.5); print(f"hieu chuan c=1,5 [{time.time()-t0:.0f}s]")
    def cv_cfg(name, key, margin, int32_ops=(), out_pc=True):
        per = {}; sim.reset_fb(); t0 = time.time()
        for test in FOLDS:
            train = [i for i in range(n) if i not in test]
            sc = sim.build_scales([STATS_C[key][i] for i in train], 100.0, 100.0, pc_margin=margin)
            set_int8(sc); sim.QS.int32_ops = tuple(int32_ops); sim.QS.out_pc = out_pc
            sim.QS.mode, sim.QS.row_fb = "quant", True
            try:
                for i in test:
                    r = REF[i]; sim.set_valid(r["inp"]["feature_attention_mask"])
                    with torch.no_grad():
                        lq, _, _ = sim.forward_logits(model, r["inp"], r["gen"])
                    m = sim.compare_logits(r["lf"], lq, r["full"], r["n0"]); per[i] = (m["top1"], m["n"], m["kl"])
            finally:
                unset_int8(); sim.QS.int32_ops = ()
        ids = sorted(per); w = np.array([per[i][1] for i in ids], dtype=float)
        t1 = np.array([per[i][0] for i in ids]); kl = np.array([per[i][2] for i in ids])
        return dict(name=name, top1=float((t1 * w).sum() / w.sum()), kl=float((kl * w).sum() / w.sum()), fb=fb_rate(), t1=t1, w=w, secs=time.time() - t0)
    CONFIGS = [("P1.5 (chuan, c=2)", 2.0, 1.5, (), True), ("P1.75, c=2", 2.0, 1.75, (), True), ("P2.0, c=2", 2.0, 2.0, (), True),
               ("P1.5, c=1.5", 1.5, 1.5, (), True), ("INT32 moi GEMM, c=2 (IO x4)", 2.0, 1.0, ("*",), False)]
    for cfg in CONFIGS:
        SWEEP.append(cv_cfg(*cfg)); s = SWEEP[-1]
        print("%-30s top1=%5.1f%%  KL=%.4f  hang-FP=%.2f%%  [%.0fs]" % (s["name"], s["top1"] * 100, s["kl"], s["fb"], s["secs"]))
    rng = np.random.default_rng(1); ref = SWEEP[0]; idx = rng.integers(0, len(ref["w"]), (5000, len(ref["w"])))
    print("\nchenh lech top-1 so voi P1.5 chuan (khoang tin cay 95%, lay mau lai theo clip):")
    for s in SWEEP[1:]:
        d = (s["t1"] - ref["t1"]); bs = np.array([(d * ref["w"])[i].sum() / ref["w"][i].sum() for i in idx])
        s["diff_vs_ref"] = [float(d @ ref["w"] / ref["w"].sum()), float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))]
        print("  %-30s %+5.2f diem  [%+5.2f; %+5.2f]" % (s["name"], s["diff_vs_ref"][0] * 100, s["diff_vs_ref"][1] * 100, s["diff_vs_ref"][2] * 100))
else:
    print("Bo qua quet them (RUN_SWEEP = False)")

Bo qua quet them (RUN_SWEEP = False)


In [18]:
# ===== 13. Gioi han 2: sinh cau tra loi bang INT8 (P1.5, co dinh) tren bo cau hoi CHUA TUNG dung de hieu chuan =====
SCALES_ALL = sim.build_scales(STATS, 100.0, 100.0, pc_margin=PC_MARGIN)
set_int8(SCALES_ALL)
print("== INT8 (P1.5, scale hieu chuan tren", len(STATS), "clip hoi thoai) ==")
try:
    quiz_run("int8")
finally:
    unset_int8()

== INT8 (P1.5, scale hieu chuan tren 50 clip hoi thoai) ==
  [int8] 5/50  31s  (con ~281s)  token=10
  [int8] 10/50  59s  (con ~235s)  token=23
  [int8] 15/50  104s  (con ~243s)  token=86
  [int8] 20/50  171s  (con ~257s)  token=82
  [int8] 25/50  229s  (con ~229s)  token=69
  [int8] 30/50  273s  (con ~182s)  token=37
  [int8] 35/50  309s  (con ~132s)  token=16
  [int8] 40/50  340s  (con ~85s)  token=60
  [int8] 45/50  396s  (con ~44s)  token=23
  [int8] 50/50  442s  (con ~0s)  token=20


In [19]:
# ===== 14. Phan tich do dung: FP32 vs moc vs INT8 (so sanh theo cap) =====
LAB = {"fp32": "FP32", "lowprec": BF_TAG, "int8": "INT8 P1.5"}
modes = [m for m in ("fp32", "lowprec", "int8") if m in QUIZ_RES]
G = {m: [grade(d["text"], QUIZ[i]["pats"]) for i, d in enumerate(QUIZ_RES[m])] for m in modes}
N = len(QUIZ)
rng = np.random.default_rng(0)
print("so cau:", N, "| cau cham tran", QUIZ_MAX_NEW, "token (bi cat):", {LAB[m]: sum(d["capped"] for d in QUIZ_RES[m]) for m in modes})
print("do dai TB (token):", {LAB[m]: round(float(np.mean([d["n_tok"] for d in QUIZ_RES[m]])), 1) for m in modes})
print("\nDo dung:  s1 = mau dung o CAU DAU (muc chinh) | s2 = trong 2 cau dau | any = bat ky dau (can tren, co the tinh nham)")
print("%-12s %16s %16s %16s" % ("", "s1", "s2", "any"))
ACC = {}
for m in modes:
    row = [sum(g[k] for g in G[m]) for k in ("s1", "s2", "any")]; ACC[m] = dict(zip(("s1", "s2", "any"), row), n=N)
    print("%-12s " % LAB[m] + " ".join("%8d/%d %4.1f%%" % (v, N, 100 * v / N) for v in row))
print()
def paired(a, b, key):
    A = np.array([g[key] for g in G[a]]); B = np.array([g[key] for g in G[b]])
    bb, cc = int(np.sum(A & ~B)), int(np.sum(~A & B))
    idx = rng.integers(0, N, (5000, N)); diff = (B.astype(float) - A.astype(float))[idx].mean(1)
    return bb, cc, mcnemar_p(bb, cc), float(np.mean(B) - np.mean(A)), np.percentile(diff, [2.5, 97.5])
PAIRS_OUT = {}
for a, b in (("fp32", "int8"), ("fp32", "lowprec"), ("lowprec", "int8")):
    if a not in modes or b not in modes: continue
    for key in ("s1", "s2", "any"):
        bb, cc, p, d, ci = paired(a, b, key); PAIRS_OUT[f"{a}->{b}:{key}"] = dict(a_right_b_wrong=bb, a_wrong_b_right=cc, p=p, diff=d, ci=[float(x) for x in ci])
        print(f"{LAB[a]:10s} -> {LAB[b]:10s} [{key:3s}]  dung->sai {bb:3d} | sai->dung {cc:3d} | McNemar p={p:.3f} | do dung {LAB[b]}-{LAB[a]}: {d*100:+.1f} diem  [{ci[0]*100:+.1f}; {ci[1]*100:+.1f}]")
print("\ntheo nhom (s1):")
for cat, _ in CATS:
    ix = [i for i, q in enumerate(QUIZ) if cat_of(q["k"]) == cat]
    if ix: print("  %-16s (n=%2d) " % (cat, len(ix)) + "  ".join("%s %d/%d" % (LAB[m], sum(G[m][i]["s1"] for i in ix), len(ix)) for m in modes))
if "fp32" in modes:
    print("\nso cau co van ban giong het FP32:", {LAB[m]: sum(QUIZ_RES["fp32"][i]["text"] == QUIZ_RES[m][i]["text"] for i in range(N)) for m in modes if m != "fp32"})
if "int8" in modes:
    fbs = [d["fb"] for d in QUIZ_RES["int8"]]; pre = [d["fb_pre"] for d in QUIZ_RES["int8"] if d["fb_pre"] is not None]; dec = [d["fb_dec"] for d in QUIZ_RES["int8"] if d["fb_dec"] is not None]
    print("\nHang FP cua INT8 khi sinh cau tra loi: TB %.2f%% (trung vi %.2f%%, nho nhat %.2f%%, lon nhat %.2f%%)" % (np.mean(fbs), np.median(fbs), np.min(fbs), np.max(fbs)))
    print("   tach: prefill TB %.2f%% | decode (1 hang/buoc) TB %.2f%%   | doi chung teacher-forced tren clip hoi thoai: %.2f%%" % (np.mean(pre) if pre else float("nan"), np.mean(dec) if dec else float("nan"), CV_FB))
if "fp32" in modes and "int8" in modes:
    print("\n=== Cau FP32 va INT8 cham KHAC nhau o BAT KY muc nao (de duyet tay; van ban day du) ===")
    for i in range(N):
        a, b = G["fp32"][i], G["int8"][i]
        if any(a[k] != b[k] for k in ("s1", "s2", "any")):
            print(f"[{QUIZ[i]['k']}] {QUIZ[i]['q']}   (mau: {QUIZ[i]['pats']})")
            print(f"   FP32 (s1={a['s1']}, any={a['any']}): {QUIZ_RES['fp32'][i]['text']}")
            print(f"   INT8 (s1={b['s1']}, any={b['any']}): {QUIZ_RES['int8'][i]['text']}")

so cau: 50 | cau cham tran 300 token (bi cat): {'FP32': 0, 'INT8 P1.5': 0}
do dai TB (token): {'FP32': 35.9, 'INT8 P1.5': 35.0}

Do dung:  s1 = mau dung o CAU DAU (muc chinh) | s2 = trong 2 cau dau | any = bat ky dau (can tren, co the tinh nham)
                           s1               s2              any
FP32               39/50 78.0%       39/50 78.0%       39/50 78.0%
INT8 P1.5          38/50 76.0%       38/50 76.0%       38/50 76.0%

FP32       -> INT8 P1.5  [s1 ]  dung->sai   3 | sai->dung   2 | McNemar p=1.000 | do dung INT8 P1.5-FP32: -2.0 diem  [-10.0; +6.0]
FP32       -> INT8 P1.5  [s2 ]  dung->sai   3 | sai->dung   2 | McNemar p=1.000 | do dung INT8 P1.5-FP32: -2.0 diem  [-10.0; +6.0]
FP32       -> INT8 P1.5  [any]  dung->sai   3 | sai->dung   2 | McNemar p=1.000 | do dung INT8 P1.5-FP32: -2.0 diem  [-10.0; +6.0]

theo nhom (s1):
  dia ly           (n= 9) FP32 8/9  INT8 P1.5 8/9
  Viet Nam         (n= 4) FP32 1/4  INT8 P1.5 1/4
  khoa hoc         (n= 9) FP32 8/9  INT8 P1.5

In [20]:
# ===== 15. Luu ket qua =====
os.makedirs(OUT_DIR, exist_ok=True)
CONFIG = {"key": 2.0, "row_c": ROW_C, "int32_ops": [], "out_pc": True, "pc_margin": PC_MARGIN, "sat": "clip", "score_mode": "int32", "lm_out": "int32", "smooth": SM_KW}
def _clean(s): return {k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in s.items()}
json.dump({"platform": PLATFORM, "n_clips": len(REF), "quiz_n": len(QUIZ), "tie": TIE, "pairs": PAIRS_OUT, "config": CONFIG, "bf_tag": BF_TAG,
           "cv_fb": CV_FB, "cv_fb_split": CV_SPLIT, "sweep": [_clean(s) for s in SWEEP],
           "quiz": [dict(k=q["k"], q=q["q"], pats=q["pats"], **{m: QUIZ_RES[m][i] for m in modes}, **{"g_" + m: G[m][i] for m in modes}) for i, q in enumerate(QUIZ)],
           "accuracy": ACC, "smooth_info": {k: list(v) for k, v in SM_INFO.items()}},
          open(f"{OUT_DIR}/quiz_report.json", "w"), indent=1, ensure_ascii=False, default=str)
json.dump({"scales": {k: (v.tolist() if torch.is_tensor(v) else v) for k, v in SCALES_ALL.items()}, "config": CONFIG, "pcts": sim.PCTS},
          open(f"{OUT_DIR}/scales_P15.json", "w"))
import shutil
shutil.make_archive(f"{ROOT}/qint8_report_v12", "zip", OUT_DIR)
print("Da luu:", OUT_DIR, "->", f"{ROOT}/qint8_report_v12.zip")
if PLATFORM == "colab" and not OUT_DIR.startswith("/content/drive"):
    print("LUU Y: chua gan Google Drive, ket qua nam o dia cuc bo va MAT khi ngat ket noi. Tai file zip ve ngay (Files -> qint8_report_v12.zip).")

Da luu: /kaggle/working/qint8_report -> /kaggle/working/qint8_report_v12.zip
